# Group 10 — COVID-19 Data Visualizations Project

**Combined Notebook**

## Contents

1. [Setup](#Setup)
2. [Part 1 — Data Wrangling](#Part-1-—-Data-Wrangling)
3. [Part 2 — Mobility Data](#Part-2-—-Mobility-Data)
4. [Part 3 — Modeling & Analysis](#Part-3-—-Modeling-&-Analysis)
5. [Part 4 — Visualizations](#Part-4-—-Visualizations)
6. [Part 5 — Dashboard](#Part-5-—-Dashboard)


## Setup

In [ ]:
# Consolidated imports for all sections
import pandas as pd
import numpy as np
from numpy import polyfit, poly1d

import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import plotly.express as px

import pycountry
import ipywidgets as widgets
from IPython.display import display, HTML
from vega_datasets import data as vega_data

from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

alt.data_transformers.disable_max_rows()


---

## Part 1 — Data Wrangling

*Source: `covidData_wrang.ipynb`*


## About the Dataset

**Dataset:** OxCGRT COVID-19 Dataset  
**Source:** https://github.com/OxCGRT  
**File:** `owid-covid-data.csv`   

In [ ]:
owid = pd.read_csv('../data/owid-covid-data.csv')
display(owid.head())

In [ ]:
# shape
print(owid.shape)
print(f'\nThere are {owid.shape[0]} rows and {owid.shape[1]} columns\n')
print('- Each row is one case of COVID')

In [ ]:
# 2. & 3 Column Names & Data Types
print("\nColumn Names and Data Types:")
for col in owid.columns:
    print(f"  - {col}: {owid[col].dtype}")

The dataset contains 67 variables including identifiers (e.g., country codes),
temporal variables (date), and epidemiological measures such as cases,
deaths, and vaccinations.

In [ ]:
classification = []

for col in owid.columns:
    if col in {'date', 'iso_code'}:
        classification.append({
            'column': col,
            'data_type': 'identifier / temporal',
            'measurement_type': 'not applicable'
        })
    elif not pd.api.types.is_numeric_dtype(owid[col]) or pd.api.types.is_bool_dtype(owid[col]):
        classification.append({
            'column': col,
            'data_type': 'categorical',
            'measurement_type': 'categorical (nominal)'
        })
    else:
        values = owid[col].dropna()
        if len(values) > 0 and ((values % 1) == 0).all():
            measurement = 'discrete'
        else:
            measurement = 'continuous'

        classification.append({
            'column': col,
            'data_type': 'quantitative',
            'measurement_type': measurement
        })

classification_df = pd.DataFrame(classification)
classification_df


Numeric variables representing counts (e.g., cases, deaths) are treated as
**discrete**, while rates and averages are treated as **continuous**.
Identifier and date columns are excluded from quantitative analysis.

In [ ]:
# categorical columns
categorical_cols = owid.select_dtypes(include=['object', 'string']).columns

categorical_values = {
    col: owid[col].dropna().unique()
    for col in categorical_cols
}

categorical_values


In [ ]:
categorical_summary = []

for col in categorical_cols:
    counts = owid[col].value_counts()
    categorical_summary.append({
        'column': col,
        'unique_values': owid[col].nunique(),
        'mode': counts.idxmax(),
        'mode_count': counts.max()
    })

pd.DataFrame(categorical_summary)

For categorical variables, the mode was identified as the most frequently
occurring category based on value counts.

In [ ]:
# numeric columns
numeric_cols = owid.select_dtypes(include='number').columns

quant_summary = []

for col in numeric_cols:
    quant_summary.append({
        'column': col,
        'min': owid[col].min(),
        'max': owid[col].max(),
        'mean': owid[col].mean(),
        'median': owid[col].median(),
        'std': owid[col].std()
    })

quant_df = pd.DataFrame(quant_summary)
quant_df

Statistics are reported only for variables where they are meaningful.
Count-based variables (e.g., cases, deaths, vaccinations) are measured in
**number of people**, while rate-based variables are measured in **percentages
or per-capita units**.

Identifier variables (e.g., codes or IDs) are excluded, as summary statistics
would be misleading.

In [ ]:
skip_cols = {'iso_code', 'date'}

summary = []

for col in owid.columns:
    missing_pct = owid[col].isna().mean() * 100

    if not pd.api.types.is_numeric_dtype(owid[col]) or pd.api.types.is_bool_dtype(owid[col]):
        summary.append({
            'Column': col,
            'Type': 'Categorical',
            'Measurement': 'Nominal',
            'Missing %': round(missing_pct, 2),
            'Details': f"Unique values: {owid[col].nunique()}, "
                       f"Mode: {owid[col].mode().iloc[0] if not owid[col].mode().empty else 'N/A'}"
        })
    else:
        values = owid[col].dropna()
        measurement = (
            'Discrete'
            if len(values) > 0 and ((values % 1) == 0).all()
            else 'Continuous'
        )

        summary.append({
            'Column': col,
            'Type': 'Quantitative',
            'Measurement': measurement,
            'Missing %': round(missing_pct, 2),
            'Details': (
                f"Range: {values.min():.2f}–{values.max():.2f}, "
                f"Mean: {values.mean():.2f}, "
                f"Median: {values.median():.2f}, "
                f"Std: {values.std():.2f}"
            )
        })

summary_df = pd.DataFrame(summary)

summary_df = summary_df.sort_values(by=['Type', 'Measurement', 'Missing %'], ascending=[True, True, False])

summary_df


The table above summarizes all variables in the dataset, including data type,
measurement scale, missingness, and relevant descriptive statistics.
Statistics are reported only where meaningful.

## Missing Value Check

In [ ]:
# Check missing values
missing_summary = owid.isna().sum().to_frame('missing_count')
missing_summary['missing_pct'] = (missing_summary['missing_count'] / len(owid) * 100).round(2)
missing_summary.sort_values('missing_pct', ascending=False)

In [ ]:
# fill categorical data with mode
categorical_cols = owid.select_dtypes(include=['object', 'string']).columns

for col in categorical_cols:
    mode = owid[col].mode()
    if not mode.empty:
        owid[col] = owid[col].fillna(mode.iloc[0])


In [ ]:
# fill numerical data with mean (if makes sense)
numeric_cols = owid.select_dtypes(include='number').columns

for col in numeric_cols:
    mean_val = owid[col].mean()
    owid[col].fillna(mean_val, inplace=True)

In [ ]:
# if a row is missing an identifier or date, drop it
owid.dropna(subset=['iso_code', 'date'], inplace=True)

In [ ]:
# Missing values per column after filling
owid.isna().sum()

## Sanity Check

In [ ]:
# Select numeric columns for correlation analysis
key_cols = owid.select_dtypes(include=['float64', 'int64']).columns.tolist()
print(f"Numeric columns for correlation ({len(key_cols)}):")
print(key_cols[:10], "...")  # Show first 10

In [ ]:
# Ensure all counts/ratios are non-negative
numeric_checks = numeric_cols[numeric_cols.str.contains('cases|deaths|tests|vaccinations|population', case=False)]
for col in numeric_checks:
    if (owid[col] < 0).any():
        print(f"Warning: negative values in {col}")

In [ ]:
# Ensure date is datetime type
owid['date'] = pd.to_datetime(owid['date'])

## Duplicate Check

In [ ]:
duplicates = owid.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

## Stats

In [ ]:
owid.describe().T

In [ ]:
correlation_matrix = owid[numeric_cols].corr(method='pearson')
print(correlation_matrix)

In [ ]:
# Select numeric columns for correlation analysis                     
key_cols = owid.select_dtypes(include=['float64', 'int64']).columns.tolist()                                                    
print(f"Numeric columns for correlation ({len(key_cols)}):")                                                                    
print(key_cols[:10], "...")

In [ ]:
corr = owid[key_cols].corr().abs()
high_corr_pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
                       .stack()
                       .sort_values(ascending=False))
high_corr_pairs = high_corr_pairs[high_corr_pairs > 0.5]
print(high_corr_pairs)

In [ ]:
## Visualization of Top Correlations

top_pairs = [
    ('total_cases', 'total_deaths', 0.946),
    ('total_vaccinations', 'people_vaccinated', 0.987),
    ('people_vaccinated', 'population', 0.723),
    ('total_cases', 'total_vaccinations', 0.682),
    ('new_cases', 'new_deaths', 0.506),
    ('total_deaths', 'population', 0.709),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (var_x, var_y, corr_value) in enumerate(top_pairs):
    ax = axes[idx]

    pair_df = owid[[var_x, var_y]].dropna()
    x = pair_df[var_x].values
    y = pair_df[var_y].values

    ax.scatter(x, y, alpha=0.15, s=8, color='steelblue')

    if len(x) >= 2 and np.ptp(x) > 0:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, p(x_line), "r-", linewidth=2.5, label='Linear Fit')
        ax.legend(loc='best', fontsize=9)

    ax.set_xlabel(var_x.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.set_ylabel(var_y.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.set_title(f'{var_x} vs {var_y}\n(r = {corr_value:.3f})', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()


In [ ]:
## Correlation Heatmap

# Select key variables related to your research
key_vars = [
    'total_cases', 'total_deaths', 'new_cases', 'new_deaths',
    'total_vaccinations', 'people_vaccinated', 'population',
    'stringency_index', 'total_deaths_per_million'
]

# Create correlation matrix for key variables
corr_matrix = owid[key_vars].corr()

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of Key COVID-19 Variables', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ Correlation heatmap created and saved as 'correlation_heatmap.png'")

In [ ]:
## Analysis: Stringency Index vs Mortality 

# Remove rows with missing stringency_index
data_with_stringency = owid[owid['stringency_index'] > 0].copy()

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Stringency vs Total Deaths Per Million
ax1 = axes[0]
ax1.scatter(data_with_stringency['stringency_index'], 
            data_with_stringency['total_deaths_per_million'],
            alpha=0.2, s=10, color='crimson')
# Add trend line
z1 = np.polyfit(data_with_stringency['stringency_index'], 
                data_with_stringency['total_deaths_per_million'], 1)
p1 = np.poly1d(z1)
x_range = np.linspace(data_with_stringency['stringency_index'].min(), 
                      data_with_stringency['stringency_index'].max(), 100)
ax1.plot(x_range, p1(x_range), 'r-', linewidth=2.5, label='Trend')
corr1 = data_with_stringency['stringency_index'].corr(data_with_stringency['total_deaths_per_million'])
ax1.set_xlabel('Stringency Index (Government Response Severity)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Total Deaths Per Million', fontsize=11, fontweight='bold')
ax1.set_title(f'Stringency vs Mortality Rate\n(r = {corr1:.3f})', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Stringency vs New Deaths Smoothed
ax2 = axes[1]
ax2.scatter(data_with_stringency['stringency_index'], 
            data_with_stringency['new_deaths_smoothed'],
            alpha=0.2, s=10, color='orange')
# Add trend line
z2 = np.polyfit(data_with_stringency['stringency_index'], 
                data_with_stringency['new_deaths_smoothed'], 1)
p2 = np.poly1d(z2)
ax2.plot(x_range, p2(x_range), 'r-', linewidth=2.5, label='Trend')
corr2 = data_with_stringency['stringency_index'].corr(data_with_stringency['new_deaths_smoothed'])
ax2.set_xlabel('Stringency Index (Government Response Severity)', fontsize=11, fontweight='bold')
ax2.set_ylabel('New Deaths (Smoothed)', fontsize=11, fontweight='bold')
ax2.set_title(f'Stringency vs New Deaths (Smoothed)\n(r = {corr2:.3f})', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✓ Stringency analysis complete")
print(f"  - Correlation (Stringency vs Deaths/Million): {corr1:.4f}")
print(f"  - Correlation (Stringency vs New Deaths): {corr2:.4f}")

## About the Dataset

**Dataset:** Oxford COVID-19 Government Tracker  
**Source:** https://github.com/OxCGRT/covid-policy-dataset/tree/main  
**File:** `OxCGRT_compact_national_v1.csv`   

In [ ]:
OxCGRT = pd.read_csv("../data/OxCGRT_compact_national_v1.csv")
OxCGRT.head()

In [ ]:
OxCGRT.tail()

In [ ]:
OxCGRT.info()

In [ ]:
OxCGRT.describe()
OxCGRT.shape

In [ ]:
# Column Names & Data Types
print("\nColumn Names and Data Types:")
for col in OxCGRT.columns:
    print(f"  - {col}: {OxCGRT[col].dtype}")

##### removing some columns to make initial statistic analysis easier


In [ ]:
# Flags variables will tell if a policy is targeted to a specific geography, or general across the whole jurisdiction.
# Also the region columns are compleatly empty
OxCGRT = OxCGRT.drop(columns=['C1M_Flag', 'C2M_Flag', 'C3M_Flag', 'C4M_Flag', 'C5M_Flag', 'C6M_Flag', 'C7M_Flag', 'E1_Flag', 'H1_Flag', 'H6M_Flag', 'H7_Flag', 'H8M_Flag', 'RegionName', 'RegionCode'])

In [ ]:
OxCGRT.head()
OxCGRT.shape

In [ ]:
# 42 columns now
# count the missing values in each column
for column in OxCGRT.columns.values.tolist():
    print(column)
    print(OxCGRT[column].isnull().sum())
    print(' ')

In [ ]:
# fill categorical data with mode
categorical_cols = OxCGRT.select_dtypes(include=['object', 'string']).columns

for col in categorical_cols:
    mode_val = OxCGRT[col].mode()[0]
    OxCGRT[col] = OxCGRT[col].fillna(mode_val)

In [ ]:
# fill numerical data with mean (if makes sense)
numeric_cols = OxCGRT.select_dtypes(include='number').columns

for col in numeric_cols:
    mean_val = OxCGRT[col].mean()
    OxCGRT[col] = OxCGRT[col].fillna(mean_val)

In [ ]:
# checking missing values
OxCGRT.isna().sum()

In [ ]:
# Select numeric columns for correlation analysis                     
key_cols = OxCGRT.select_dtypes(include=['float64', 'int64']).columns.tolist()                                                    
print(f"Numeric columns for correlation ({len(key_cols)}):")                                                                    
print(key_cols[:10], "...")

In [ ]:
## Correlation Heatmap

# Select key variables related to your research
key_vars = [
    'C1M_School closing', 'C2M_Workplace closing', 'C3M_Cancel public events', 
    'C4M_Restrictions on gatherings', 'C5M_Close public transport', 'C6M_Stay at home requirements', 
    'C7M_Restrictions on internal movement', 'C8EV_International travel controls', 'E1_Income support'
]

# Create correlation matrix for key variables
corr_matrix = OxCGRT[key_vars].corr()

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of Key COVID-19 Variables', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


## Merging the Data!

In [ ]:
# Find matching columns
matching_cols = set(owid.columns) & set(OxCGRT.columns)
print(f"\n=== MATCHING COLUMNS ({len(matching_cols)}) ===")
if matching_cols:
   for col in sorted(matching_cols):
       print(f"  - {col}")
else:
   print("  No matching column names found")

In [ ]:
# Prepare datasets for merging
owid_merge = owid[['location', 'date', 'total_deaths', 'total_cases', 'population']].copy()
oxford_merge = OxCGRT[['CountryName', 'Date', 'StringencyIndex_Average']].copy()

# Rename for consistent merging
owid_merge.rename(columns={'location': 'country'}, inplace=True)
oxford_merge.rename(columns={'CountryName': 'country'}, inplace=True)

# Fix date formats
owid_merge['date'] = pd.to_datetime(owid_merge['date'])
# Oxford dates are integers like 20200101 - convert them properly
oxford_merge['Date'] = oxford_merge['Date'].astype(str).str.replace('.', '').str[:8]
oxford_merge['date'] = pd.to_datetime(oxford_merge['Date'], format='%Y%m%d')
oxford_merge = oxford_merge.drop('Date', axis=1)

# Calculate mortality rate (deaths per 1000 population)
owid_merge['mortality_rate'] = (owid_merge['total_deaths'] / owid_merge['population']) * 1000

print(f"OWID shape: {owid_merge.shape}")
print(f"Oxford shape: {oxford_merge.shape}")
print(f"\nOWID date range: {owid_merge['date'].min()} to {owid_merge['date'].max()}")
print(f"Oxford date range: {oxford_merge['date'].min()} to {oxford_merge['date'].max()}")

In [ ]:
# Merge datasets on country and date
merged_data = pd.merge(
    owid_merge, 
    oxford_merge, 
    on=['country', 'date'], 
    how='inner'  # Only keep rows where both datasets have data
)

print(f"Merged data shape: {merged_data.shape}")
print(f"\nMissing values in merged data:")
print(merged_data.isnull().sum())
print(f"\nSample of merged data:")
print(merged_data.head(10))

In [ ]:
# Clean merged data - remove rows with missing critical values
analysis_data = merged_data.dropna(subset=['mortality_rate', 'StringencyIndex_Average'])

print(f"Data for analysis shape: {analysis_data.shape}")
print(f"Countries in analysis: {analysis_data['country'].nunique()}")
print(f"Date range: {analysis_data['date'].min()} to {analysis_data['date'].max()}")

# Summary statistics
print(f"\nMortality Rate (per 1000 population):")
print(analysis_data['mortality_rate'].describe())
print(f"\nStringency Index:")
print(analysis_data['StringencyIndex_Average'].describe())

In [ ]:
# Sanity check for merged analysis data
print("=== MERGED DATA INFO ===")
print(f"Shape: {analysis_data.shape}")
print(f"\nData types:")
print(analysis_data.dtypes)
print(f"\n=== MISSING VALUES ===")
print(analysis_data.isnull().sum())
print(f"\n=== STATISTICS ===")
print(analysis_data.describe())

In [ ]:
# Analyze correlation between mortality rate and stringency
correlation = analysis_data['mortality_rate'].corr(analysis_data['StringencyIndex_Average'])
print(f"Correlation between Mortality Rate and Stringency Index: {correlation:.4f}")

# Create scatter plot
plt.figure(figsize=(12, 6))
plt.scatter(analysis_data['StringencyIndex_Average'], analysis_data['mortality_rate'], 
            alpha=0.3, s=20)
plt.xlabel('Stringency Index')
plt.ylabel('Mortality Rate (per 1000 population)')
plt.title(f'Mortality Rate vs Stringency Index\n(Correlation: {correlation:.4f})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Build combined df: owid columns + merged stringency/mortality
df = owid.merge(
    analysis_data[['country', 'date', 'StringencyIndex_Average', 'mortality_rate']],
    left_on=['location', 'date'],
    right_on=['country', 'date'],
    how='left'
)

# PEARSON CORRELATIONS
print("PEARSON CORRELATION COEFFICIENTS")

numeric_cols = df.select_dtypes(include='number').columns
corr = df[numeric_cols].corr(method='pearson').abs()
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack()
             .sort_values(ascending=False))
high = pairs[pairs > 0.5]
print(high.to_string())
print()


# POLYNOMIAL FIT COMPARISON
print("POLYNOMIAL FIT COMPARISON (Linear / Quadratic / Cubic)")

top_pairs = [
    ('total_cases',          'total_deaths'),
    ('total_vaccinations',   'people_vaccinated'),
    ('people_vaccinated',    'total_cases'),
    ('new_cases',            'new_deaths'),
    ('total_deaths',         'total_cases'),
    ('StringencyIndex_Average', 'mortality_rate'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

results = []

for idx, (x_col, y_col) in enumerate(top_pairs):
    pair_df = df[[x_col, y_col]].dropna()
    x = pair_df[x_col].values
    y = pair_df[y_col].values

    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]

    ax = axes[idx]
    if len(x) < 4 or np.ptp(x) == 0:
        ax.text(0.5, 0.5, f'Not enough data\nfor {x_col} vs {y_col}',
                ha='center', va='center', transform=ax.transAxes)
        continue

    r2_scores = {}
    ax.scatter(x, y, alpha=0.1, s=6, color='steelblue', label='Data')

    x_line = np.linspace(x.min(), x.max(), 300)
    colors = {'Linear':'red', 'Quadratic':'orange', 'Cubic':'green'}

    for degree, label in [(1,'Linear'),(2,'Quadratic'),(3,'Cubic')]:
        coeffs = polyfit(x, y, degree)
        p      = poly1d(coeffs)
        y_pred = p(x)
        r2     = r2_score(y, y_pred)
        r2_scores[label] = round(r2, 4)
        ax.plot(x_line, p(x_line), color=colors[label], linewidth=2, label=f'{label} R²={r2:.4f}')

    best = max(r2_scores, key=r2_scores.get)
    ax.set_xlabel(x_col.replace('_',' ').title(), fontsize=9, fontweight='bold')
    ax.set_ylabel(y_col.replace('_',' ').title(), fontsize=9, fontweight='bold')
    ax.set_title(f'{x_col} vs {y_col}\nBest fit: {best}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    results.append({
        'X': x_col, 'Y': y_col,
        'Linear R²': r2_scores['Linear'],
        'Quadratic R²': r2_scores['Quadratic'],
        'Cubic R²': r2_scores['Cubic'],
        'Best Fit': best
    })
    print(f"{x_col} vs {y_col}:")
    for k,v in r2_scores.items():
        print(f"   {k}: R² = {v}")
    print(f"   → Best fit: {best}\n")

plt.suptitle('Polynomial Fit Comparison: Linear vs Quadratic vs Cubic', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print()


---

## Part 2 — Mobility Data

*Source: `V5_MobilityData.ipynb`*


In [ ]:
df = pd.read_csv("../data/mobility_report_countries.csv")

In [ ]:
df.dtypes ## change the date column to datetime and make it the index, maybe aggregate by week? make sure it follows the stringency index

In [ ]:

#df = df[(df["country"] == "United States") | (df["country"] == "Singapore") | (df["country"] == "South Korea")]
df = df[df["region"] == "Total"]
df.head()

In [ ]:
df.head() ## date from 2/15/2020 to 8/19/2022

In [ ]:
## want changes in mobility, we care when mobility as a whole increases or decreases
import datetime

df.reset_index()

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df.info()

In [ ]:
out_cols = [
    'retail and recreation',
    'grocery and pharmacy',
    'parks',
    'transit stations',
    'workplaces'
]

df['total_out'] = df[out_cols].mean(axis=1) 

In [ ]:
df['week'] = df['date'].dt.to_period('W')

In [ ]:
weekly_df = df.groupby(['week',"country"]).agg({
    'total_out': 'mean',
    'residential': 'mean'
}).reset_index()

In [ ]:
weekly_df

In [ ]:
s = pd.read_csv("../data/OxCGRT_compact_national_v1.csv")

In [ ]:
s.head()

In [ ]:
s['Date'] = pd.to_datetime(s['Date'], format='%Y%m%d')

In [ ]:
strin = s.copy()
strin['week'] = strin['Date'].dt.to_period('W')
strin.head()

In [ ]:
weekly_stringency = strin.groupby(['week', "CountryName"])['StringencyIndex_Average'].mean().reset_index()

In [ ]:
weekly_stringency.head(20)

In [ ]:
import altair as alt
countries = [
    "United States",
    "China",
    "New Zealand",
    "Sweden",
    "Brazil",
    "Germany"
]

# Make copies so you do not overwrite your originals
mob = weekly_df.copy()
stri = weekly_stringency.copy()

# Convert week to datetime if still Period
if str(mob["week"].dtype).startswith("period"):
    mob["week"] = mob["week"].dt.start_time

if str(stri["week"].dtype).startswith("period"):
    stri["week"] = stri["week"].dt.start_time

# Rename country column in stringency df so both match
stri = stri.rename(columns={"CountryName": "country"})

# Filter to selected countries
mob = mob[mob["country"].isin(countries)]
stri = stri[stri["country"].isin(countries)]

# Keep only the columns we need
mob = mob[["week", "country", "residential"]]
stri = stri[["week", "country", "StringencyIndex_Average"]]

# Merge
final_df = pd.merge(mob, stri, on=["week", "country"], how="inner")

# Reshape to long format for two-line plotting
plot_df = final_df.melt(
    id_vars=["week", "country"],
    value_vars=["residential", "StringencyIndex_Average"],
    var_name="measure",
    value_name="value"
)

# Cleaner labels
plot_df["measure"] = plot_df["measure"].replace({
    "residential": "Residential Mobility",
    "StringencyIndex_Average": "Stringency Index"
})

# Faceted line chart with shared axes
chart = alt.Chart(plot_df).mark_line().encode(
    x=alt.X("week:T", title="Week"),
    y=alt.Y(
        "value:Q",
        title="Value",
        scale=alt.Scale(domain=[plot_df["value"].min(), plot_df["value"].max()])
    ),
    color=alt.Color("measure:N", title="Measure"),
    tooltip=[
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("measure:N", title="Measure"),
        alt.Tooltip("value:Q", title="Value", format=".2f")
    ]
).properties(
    width=250,
    height=180
).facet(
    column=alt.Column("country:N", title="Country")
).properties(
    title="Weekly Stringency Index and Residential Mobility by Country"
)

chart

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.enable("vegafusion")

# Event dates
events = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',       'y': 75},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',         'y': 68},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',          'y': 56},
    {'date': '2020-06-08', 'label': 'NZ Reopens',           'y': 70},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',      'y': 72},
    {'date': '2021-07-01', 'label': 'Delta Wave',           'y': 66},
    {'date': '2021-12-01', 'label': 'Omicron Wave',         'y': 68},
    {'date': '2022-03-01', 'label': 'Restrictions Lifted',  'y': 72},
])

countries = [
    "United States",
    "New Zealand",
    "Sweden",
    "Brazil",
    "Germany"
]

# Use ALL countries, not just the highlighted ones
plot_df = weekly_df.copy()

# Make sure week is datetime
if str(plot_df["week"].dtype).startswith("period"):
    plot_df["week"] = plot_df["week"].dt.start_time
else:
    plot_df["week"] = pd.to_datetime(plot_df["week"])

events["date"] = pd.to_datetime(events["date"])

# Flag highlighted countries
plot_df["highlight"] = plot_df["country"].isin(countries)

# Optional: fixed x-axis so the plot does not rescale
x_min = plot_df["week"].min()
x_max = plot_df["week"].max()

# Smoothed chart source
smooth_chart = alt.Chart(plot_df).transform_window(
    smoothed_residential="mean(residential)",
    frame=[-2, 2],
    groupby=["country"]
)

# Background lines for all countries
lines = smooth_chart.mark_line().encode(
    x=alt.X(
        "week:T",
        title="Week",
        scale=alt.Scale(domain=[x_min, x_max])
    ),
    y=alt.Y(
        "smoothed_residential:Q",
        title="Residential Mobility (% Change from Baseline)",
        scale = alt.Scale(domain = [-15,80])
    ),
    detail="country:N",
    color=alt.condition(
        alt.datum.highlight,
        alt.Color(
            "country:N",
            title="Highlighted Countries",
            scale=alt.Scale(
                domain=countries,
                range=["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]
            )
        ),
        alt.value("lightgray")
    ),
    size=alt.condition(alt.datum.highlight, alt.value(2.5), alt.value(1)),
    opacity=alt.condition(alt.datum.highlight, alt.value(1.0), alt.value(0.5)),
    tooltip=[
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("smoothed_residential:Q", title="Smoothed Residential", format=".2f")
    ]
)

# Invisible points for cleaner tooltip interaction
points = smooth_chart.mark_circle(size=50, opacity=0).encode(
    x="week:T",
    y="smoothed_residential:Q",
    detail="country:N",
    tooltip=[
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("smoothed_residential:Q", title="Smoothed Residential", format=".2f")
    ]
)

# Vertical event lines
event_rules = alt.Chart(events).mark_rule(strokeDash=[4, 4], size=1).encode(
    x="date:T"
)

# Event labels
event_text = alt.Chart(events).mark_text(
    angle=0,
    align="left",
    dx=5,
    dy=-5,
    fontSize=11
).encode(
    x="date:T",
    y=alt.Y("y:Q"),
    text="label:N"
)

chart = (lines + points + event_rules + event_text).properties(
    title="Smoothed Residential Mobility Over Time Across Countries",
    width=850,
    height=450
)

chart

In [ ]:
# Convert Period → Timestamp
if str(final_df["week"].dtype).startswith("period"):
    final_df["week"] = final_df["week"].dt.to_timestamp()
else:
    final_df["week"] = pd.to_datetime(final_df["week"])

# Clean data
plot_df = final_df.dropna(subset=["country", "week", "residential", "StringencyIndex_Average"]).copy()
plot_df["week"] = pd.to_datetime(plot_df["week"])

# ---- Compute correlation in pandas (stable fix) ----
corr_df = (
    plot_df.groupby("country")
    .apply(lambda x: x["StringencyIndex_Average"].corr(x["residential"]))
    .reset_index(name="corr_value")
)

# Merge correlation back
plot_df = plot_df.merge(corr_df, on="country", how="left")

# ---- Dropdown ----
country_list = sorted(plot_df["country"].unique())

country_select = alt.param(
    name="SelectedCountry",
    bind=alt.binding_select(options=country_list, name="Choose a country: "),
    value=country_list[0]
)

# ---- Base filtered chart ----
base = alt.Chart(plot_df).transform_filter(
    "datum.country == SelectedCountry"
)

# ---- Scatter ----
points = base.mark_circle(size=70, opacity=0.6).encode(
    x=alt.X("StringencyIndex_Average:Q", title="Stringency Index"),
    y=alt.Y("residential:Q", title="Residential Mobility (% Change from Baseline)"),
    tooltip=[
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("StringencyIndex_Average:Q", title="Stringency", format=".2f"),
        alt.Tooltip("residential:Q", title="Residential", format=".2f"),
    ]
)

# ---- Regression line ----
trend = base.transform_regression(
    "StringencyIndex_Average",
    "residential"
).mark_line(size=3, color="black").encode(
    x="StringencyIndex_Average:Q",
    y="residential:Q"
)

# ---- R² ----
r2_text = base.transform_regression(
    "StringencyIndex_Average",
    "residential",
    params=True
).transform_calculate(
    label="'R² = ' + format(datum.rSquared, '.2f')"
).mark_text(
    align="left",
    baseline="top",
    fontSize=13
).encode(
    x=alt.value(10),
    y=alt.value(10),
    text="label:N"
)

# ---- Correlation ----
corr_text = base.transform_aggregate(
    corr_value="max(corr_value)"
).transform_calculate(
    label="'Correlation = ' + format(datum.corr_value, '.2f')"
).mark_text(
    align="left",
    baseline="top",
    fontSize=13
).encode(
    x=alt.value(10),
    y=alt.value(30),
    text="label:N"
)

# ---- Final chart ----
chart = (points + trend + r2_text + corr_text).add_params(
    country_select
).properties(
    width=500,
    height=400,
    title="Residential Mobility vs Stringency Index (Select Country)"
)

chart

## Visualization #6:

#### Linear Regression:

In [ ]:
s.head() ## gov support, stringency, response, containment, economic support

In [ ]:
owid = pd.read_csv("../data/owid-covid-data.csv")
owid.columns
### total_cases_per_million
### total_deaths_per_million

##### plan:
y = new_deaths_per_million

X = [["stringency_index", "residential mobility"]]

other factors: median_age, aged_65_older

health systems: hospital_beds_per_thousand

economic: gdp_per_capita

population: population_density

In [ ]:
## Location (change to country)
## date (change to datetime)
## stringency (stringency_index)
## new_deaths_per_million

### then convert to weekly

owid["date"] = pd.to_datetime(owid["date"])
owid_man = owid.copy()

cols = ["location", "date", "stringency_index", 'new_deaths_per_million']
owid_man = owid_man.loc[:, cols]
owid_man

In [ ]:
# % missing per column per country
missing_by_country = owid.groupby('location').apply(lambda x: x.isna().mean())

missing_by_country.head(20)

In [ ]:
key_cols = [
    'new_deaths_smoothed_per_million',
    'stringency_index',
    'gdp_per_capita',
    'median_age',
    'hospital_beds_per_thousand'
]

missing_by_country = owid.groupby('location')[key_cols].apply(lambda x: x.isna().mean())
missing_by_country.head(20)

In [ ]:
selected_countries = [
    "United States", "China", "New Zealand", "Sweden", "Brazil", "Germany",
    "United Kingdom", "France", "Italy", "Spain",
    "Canada", "Australia", "Japan", "South Korea", "Singapore",
    "Netherlands", "Belgium", "Denmark", "Norway"
]
owid_filter = owid[owid['location'].isin(selected_countries)].copy()

In [ ]:
print("Number of countries:", owid_filter['location'].nunique())
print(owid_filter['location'].unique())

In [ ]:
owid_filter = owid_filter[~owid_filter['iso_code'].str.startswith('OWID')]

In [ ]:
cols_needed = [
    'location', 'date',
    'new_deaths_smoothed_per_million',
    'stringency_index',
    'gdp_per_capita',
    'median_age',
    'hospital_beds_per_thousand',
    'population_density',
    'handwashing_facilities',
    'human_development_index',
    'aged_65_older',
    'aged_70_older',
    'extreme_poverty',
    'people_vaccinated_per_hundred'
]

owid_filter = owid_filter[cols_needed]
# Fill pre-vaccine era with 0
owid_filter['people_vaccinated_per_hundred'] = owid_filter['people_vaccinated_per_hundred'].fillna(0)
owid_filter.info()
owid_filter.isna().sum()

In [ ]:
owid_filter = owid_filter.dropna(subset=['new_deaths_smoothed_per_million'])
owid_filter.shape

In [ ]:
owid_filter = owid_filter.sort_values(['location', 'date'])

owid_filter['stringency_index'] = owid_filter.groupby('location')['stringency_index'].ffill()

In [ ]:
owid_filter.isna().sum()
owid_filter = owid_filter.dropna(subset=[
    'new_deaths_smoothed_per_million',
    'stringency_index',
    'gdp_per_capita',
    'median_age',
    'hospital_beds_per_thousand',
    'population_density'
])
owid_filter.shape

#### Lag/Timing Variables

In [ ]:
owid_filter = owid_filter.sort_values(["location", "date"])
owid_filter['stringency_lag_2'] = owid_filter.groupby('location')['stringency_index'].shift(2)
owid_filter['stringency_lag_3'] = owid_filter.groupby('location')['stringency_index'].shift(3)

In [ ]:
owid_filter = owid_filter.dropna(subset=['stringency_lag_2'])

In [ ]:
# Mean stringency per location (used as a country-level policy feature)
early_stringency = (
    owid_filter.groupby('location')['stringency_index']
    .mean()
    .reset_index()
    .rename(columns={'stringency_index': 'early_stringency'})
)
owid_filter = owid_filter.merge(early_stringency, on='location', how='left')

In [ ]:
core_cols = [
    'new_deaths_smoothed_per_million',
    'stringency_lag_2',
    'early_stringency',
    'gdp_per_capita',
    'median_age',
    'hospital_beds_per_thousand',
    'population_density'
]

all_cols = core_cols + [
    'location', 'date',
    'handwashing_facilities', 'human_development_index',
    'aged_65_older', 'aged_70_older',
    'extreme_poverty', 'people_vaccinated_per_hundred'
]

df_model = owid_filter[all_cols].dropna(subset=core_cols)
print(df_model.shape)

In [ ]:
import statsmodels.api as sm

X = df_model['stringency_lag_2']
y = df_model['new_deaths_smoothed_per_million']

X = sm.add_constant(X)

model1 = sm.OLS(y, X).fit()
print(model1.summary())

In [ ]:
X = df_model[
    [
        'stringency_lag_2',
        'early_stringency',
        'gdp_per_capita',
        'median_age',
        'hospital_beds_per_thousand',
        'population_density'
    ]
]

y = df_model['new_deaths_smoothed_per_million']

X = sm.add_constant(X)

model2 = sm.OLS(y, X).fit()
print(model2.summary())

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

features = [
    'stringency_lag_2',
    'early_stringency',
    'gdp_per_capita',
    'median_age',
    'hospital_beds_per_thousand',
    'population_density',
    'people_vaccinated_per_hundred'
]

df_sklearn = df_model[features + ['new_deaths_smoothed_per_million']].dropna()
print(f'Rows available for sklearn models: {len(df_sklearn)}')

X = df_sklearn[features]
y = df_sklearn['new_deaths_smoothed_per_million']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Simple OLS (sklearn)
lr_simple = LinearRegression()
lr_simple.fit(X_train[['stringency_lag_2']], y_train)

# Full OLS (sklearn)
lr_full = LinearRegression()
lr_full.fit(X_train, y_train)

# Decision Tree
tree = DecisionTreeRegressor(max_depth=4)
tree.fit(X_train, y_train)

print('Train R²:', tree.score(X_train, y_train))
print('Test R²:', tree.score(X_test, y_test))

In [ ]:
# Readable feature labels
label_map = {
    'stringency_lag_2': 'Policy Stringency (2-wk lag)',
    'early_stringency': 'Early Policy Response',
    'gdp_per_capita': 'GDP per Capita',
    'median_age': 'Median Age',
    'hospital_beds_per_thousand': 'Hospital Beds per 1k',
    'population_density': 'Population Density',
    'people_vaccinated_per_hundred': 'Vaccination Rate'
}

# --- Decision Tree: ranked feature importance ---
coef_tree = pd.DataFrame({
    'Feature': [label_map[f] for f in features],
    'Importance': list(tree.feature_importances_)
}).sort_values('Importance', ascending=False)

importance_chart = alt.Chart(coef_tree).mark_bar().encode(
    x=alt.X('Importance:Q', title='Feature Importance', axis=alt.Axis(format='.0%')),
    y=alt.Y('Feature:N', sort='-x', title=None),
    color=alt.Color('Importance:Q',
                    scale=alt.Scale(scheme='orangered'),
                    legend=None),
    tooltip=[alt.Tooltip('Feature:N'), alt.Tooltip('Importance:Q', format='.1%')]
).properties(
    title=alt.TitleParams('Decision Tree: Feature Importance', fontSize=13),
    width=320, height=200
)

importance_text = importance_chart.mark_text(
    align='left', dx=4, fontSize=11
).encode(
    text=alt.Text('Importance:Q', format='.1%'),
    color=alt.value('black')
)

importance_plot = importance_chart + importance_text

# --- OLS Full: coefficient direction (which features raise vs lower deaths) ---
coef_full = pd.DataFrame({
    'Feature': [label_map[f] for f in features],
    'Coefficient': list(lr_full.coef_)
}).sort_values('Coefficient')

coef_full['Direction'] = coef_full['Coefficient'].apply(
    lambda c: 'Increases Deaths' if c > 0 else 'Decreases Deaths'
)

ols_chart = alt.Chart(coef_full).mark_bar().encode(
    x=alt.X('Coefficient:Q', title='OLS Coefficient'),
    y=alt.Y('Feature:N', sort=alt.EncodingSortField('Coefficient', order='ascending'), title=None),
    color=alt.Color('Direction:N',
                    scale=alt.Scale(domain=['Increases Deaths', 'Decreases Deaths'],
                                    range=['#d73027', '#4575b4']),
                    legend=alt.Legend(title=None, orient='bottom')),
    tooltip=[alt.Tooltip('Feature:N'), alt.Tooltip('Coefficient:Q', format='.4f'), 'Direction:N']
).properties(
    title=alt.TitleParams('OLS Full Model: Effect Direction on Mortality', fontSize=13),
    width=320, height=200
)

rule = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='black', strokeWidth=1).encode(
    x='x:Q'
)

ols_plot = ols_chart + rule

(importance_plot | ols_plot).properties(
    title=alt.TitleParams(
        'What Predicts COVID-19 Mortality?',
        fontSize=16, anchor='middle', dy=-10
    )
)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

alt.data_transformers.disable_max_rows()

models = {
    'OLS (Simple)': lr_simple.predict(X_test[['stringency_lag_2']]),
    'OLS (Full)': lr_full.predict(X_test),
    'Decision Tree': tree.predict(X_test)
}

# --- R² bar chart with value labels ---
r2_data = pd.DataFrame([
    {'Model': name, 'R²': r2_score(y_test, preds)}
    for name, preds in models.items()
]).sort_values('R²', ascending=False)

r2_bars = alt.Chart(r2_data).mark_bar(size=40).encode(
    x=alt.X('Model:N', sort='-y', title=None, axis=alt.Axis(labelFontSize=12)),
    y=alt.Y('R²:Q', scale=alt.Scale(domain=[0, 1]), title='R² Score'),
    color=alt.Color('Model:N',
                    scale=alt.Scale(scheme='tableau10'),
                    legend=None),
    tooltip=[alt.Tooltip('Model:N'), alt.Tooltip('R²:Q', format='.3f')]
).properties(
    title=alt.TitleParams('Model R² (Test Set)', fontSize=13),
    width=220, height=200
)

r2_text = r2_bars.mark_text(dy=-8, fontSize=12, fontWeight='bold').encode(
    text=alt.Text('R²:Q', format='.2f'),
    color=alt.value('black')
)

r2_plot = r2_bars + r2_text

# --- Predicted vs Actual for Decision Tree (best model) ---
pred_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': tree.predict(X_test)
})

scatter = alt.Chart(pred_df).mark_point(opacity=0.5, size=40, color='#e6550d').encode(
    x=alt.X('Actual:Q', title='Actual Deaths per Million'),
    y=alt.Y('Predicted:Q', title='Predicted Deaths per Million'),
    tooltip=[alt.Tooltip('Actual:Q', format='.2f'), alt.Tooltip('Predicted:Q', format='.2f')]
)

max_val = float(max(pred_df['Actual'].max(), pred_df['Predicted'].max()))
diag = alt.Chart(pd.DataFrame({'v': [0, max_val]})).mark_line(
    color='steelblue', strokeDash=[5, 3], strokeWidth=2
).encode(x='v:Q', y='v:Q')

pred_plot = (scatter + diag).properties(
    title=alt.TitleParams('Decision Tree: Predicted vs Actual', fontSize=13),
    width=280, height=200
)

(r2_plot | pred_plot).properties(
    title=alt.TitleParams(
        'Model Performance on Test Set',
        fontSize=16, anchor='middle', dy=-10
    )
)

---

## Part 3 — Modeling & Analysis

*Source: `V6.ipynb`*


In [ ]:
owid = pd.read_csv("../data/owid-covid-data.csv")
owid["date"] = pd.to_datetime(owid["date"])

In [ ]:
selected_countries = [
    "United States", "China", "New Zealand", "Sweden", "Brazil", "Germany",
    "United Kingdom", "France", "Italy", "Spain",
    "Canada", "Australia", "Japan", "South Korea", "Singapore",
    "Netherlands", "Belgium", "Denmark", "Norway"
]
owid_filter = owid[owid['location'].isin(selected_countries)].copy()

owid_filter = owid_filter[~owid_filter['iso_code'].str.startswith('OWID')]

owid_filter["week"] = owid_filter["date"].dt.to_period("W").dt.start_time
owid_filter = owid_filter.sort_values(["location", "date"])


## average variables:
avg_vars = [
    "new_cases_smoothed_per_million",
    "new_deaths_smoothed_per_million",
    "reproduction_rate",
    "positive_rate",
    "stringency_index",
    "icu_patients_per_million",
    "hosp_patients_per_million",
    "weekly_icu_admissions_per_million"
]

## last value variables
last_vars = [
    "people_vaccinated_per_hundred",
    "people_fully_vaccinated_per_hundred",
    "total_boosters_per_hundred",
    "total_vaccinations_per_hundred"
]

## static:
static_vars = [
    "population_density",
    "median_age",
    "aged_65_older",
    "aged_70_older",
    "gdp_per_capita",
    "extreme_poverty",
    "cardiovasc_death_rate",
    "diabetes_prevalence",
    "female_smokers",
    "male_smokers",
    "handwashing_facilities",
    "hospital_beds_per_thousand",
    "life_expectancy",
    "human_development_index"
]

agg_dict = {}

# Average
for col in avg_vars:
    agg_dict[col] = "mean"

# Last value
for col in last_vars:
    agg_dict[col] = "last"

# Static (just take first value per country)
for col in static_vars:
    agg_dict[col] = "first"

weeklyowid_df = (
    owid_filter
    .groupby(["location", "week"])
    .agg(agg_dict)
    .reset_index()
)

weeklyowid_df = weeklyowid_df.rename(columns={"location": "country"})

In [ ]:
weeklyowid_df = weeklyowid_df.sort_values(["country", "week"])
weeklyowid_df[avg_vars + last_vars] = (
    weeklyowid_df.groupby("country")[avg_vars + last_vars].ffill()
)

In [ ]:
weeklyowid_df = weeklyowid_df.dropna(subset=["new_deaths_smoothed_per_million"])
weeklyowid_df["stringency_lag_2"] = weeklyowid_df.groupby("country")["stringency_index"].shift(2)
weeklyowid_df["stringency_lag_3"] = weeklyowid_df.groupby("country")["stringency_index"].shift(3)
weeklyowid_df["stringency_lag_4"] = weeklyowid_df.groupby("country")["stringency_index"].shift(4)

In [ ]:
weeklyowid_df.columns
(weeklyowid_df.isna().mean() * 100).round(2)

In [ ]:
# Make a copy
model_df = weeklyowid_df.copy()

# Sort first
model_df = model_df.sort_values(["country", "week"])

# 1. Drop variables with too much missingness / weaker value
drop_cols = [
    "handwashing_facilities",
    "icu_patients_per_million",
    "hosp_patients_per_million",
    "extreme_poverty"
]

model_df = model_df.drop(columns=drop_cols, errors="ignore")

# 2. Forward fill important time-series variables within each country
fill_cols = [
    "reproduction_rate",
    "positive_rate",
    "people_vaccinated_per_hundred",
    "people_fully_vaccinated_per_hundred",
    "total_vaccinations_per_hundred",
    "total_boosters_per_hundred",
    "stringency_index"
]

fill_cols = [col for col in fill_cols if col in model_df.columns]

model_df[fill_cols] = (
    model_df.groupby("country")[fill_cols].ffill()
)

# 3. Recreate lag variables after sorting/filling
model_df["stringency_lag_2"] = model_df.groupby("country")["stringency_index"].shift(2)
model_df["stringency_lag_3"] = model_df.groupby("country")["stringency_index"].shift(3)
model_df["stringency_lag_4"] = model_df.groupby("country")["stringency_index"].shift(4)

# 4. Drop rows missing target
model_df = model_df.dropna(subset=["new_deaths_smoothed_per_million"])

# 5. Optional: keep only columns you want for modeling
model_vars = [
    "country",
    "week",
    "new_deaths_smoothed_per_million",
    "new_cases_smoothed_per_million",
    "reproduction_rate",
    "positive_rate",
    "stringency_index",
    "stringency_lag_2",
    "stringency_lag_3",
    "stringency_lag_4",
    "people_vaccinated_per_hundred",
    "people_fully_vaccinated_per_hundred",
    "total_vaccinations_per_hundred",
    "total_boosters_per_hundred",
    "population_density",
    "median_age",
    "aged_65_older",
    "aged_70_older",
    "gdp_per_capita",
    "cardiovasc_death_rate",
    "diabetes_prevalence",
    "female_smokers",
    "male_smokers",
    "hospital_beds_per_thousand",
    "life_expectancy",
    "human_development_index"
]

model_vars = [col for col in model_vars if col in model_df.columns]
model_df = model_df[model_vars]

# 6. Check remaining missingness
missing_summary = pd.DataFrame({
    "Missing Count": model_df.isna().sum(),
    "Missing %": model_df.isna().mean() * 100
}).sort_values(by="Missing %", ascending=False)

print(model_df.shape)
display(missing_summary)
display(model_df.head())

In [ ]:
model_df = model_df.drop(columns=["positive_rate"], errors="ignore")
vax_cols = [
    "people_vaccinated_per_hundred",
    "people_fully_vaccinated_per_hundred",
    "total_vaccinations_per_hundred",
    "total_boosters_per_hundred"
]

# Only keep columns that exist (safe)
vax_cols = [col for col in vax_cols if col in model_df.columns]

model_df[vax_cols] = model_df[vax_cols].fillna(0)
model_df.head(20)

In [ ]:
(model_df.isna().mean() * 100).round(2)

In [ ]:
cols_to_drop = [
    "stringency_index",
    "stringency_lag_4",
    "stringency_lag_3",
    "people_fully_vaccinated_per_hundred",
    "total_vaccinations_per_hundred",
    "total_boosters_per_hundred",
    "median_age",
    "aged_70_older", 
    "cardiovasc_death_rate", 
    "female_smokers",
    "male_smokers",
    "gdp_per_capita"
]

model_df = model_df.drop(columns=cols_to_drop, errors="ignore")
final_model_df = model_df.dropna().copy()

In [ ]:
y = final_model_df["new_deaths_smoothed_per_million"]

X = final_model_df.drop(columns=[
    "new_deaths_smoothed_per_million",
    "country",
    "week"
])

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_scaled, y)

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", ascending=False)

coef_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

coef_df = pd.DataFrame({
    'Feature': [
        'stringency_lag_2',
        'new_cases_smoothed_per_million',
        'aged_65_older',
        'human_development_index',
        'population_density',
        'reproduction_rate',
        'diabetes_prevalence',
        'hospital_beds_per_thousand',
        'people_vaccinated_per_hundred',
        'life_expectancy'
    ],
    'Coefficient': [
        0.585307,
        0.577344,
        0.370707,
        0.294598,
        0.137943,
        -0.225014,
        -0.255247,
        -0.304151,
        -0.344916,
        -0.502467
    ]
})

coef_df = coef_df.sort_values(by='Coefficient')

plt.figure(figsize=(10,6))
plt.barh(coef_df['Feature'], coef_df['Coefficient'])
plt.axvline(0)
plt.title("Linear Regression Coefficients")
plt.xlabel("Effect on Deaths")
plt.show()

In [ ]:
from sklearn.metrics import r2_score

y_pred = model.predict(X_scaled)

print("R²:", r2_score(y, y_pred))

In [ ]:
# ===============================
# TIME-BASED TRAIN / TEST SPLIT + RANDOM FOREST
# ===============================

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# 1. Sort data by country and time
final_model_df = final_model_df.sort_values(["country", "week"])

# 2. Create time-based split (80% train, 20% test)
split_date = final_model_df["week"].quantile(0.8)

train_df = final_model_df[final_model_df["week"] <= split_date]
test_df  = final_model_df[final_model_df["week"] > split_date]

# 3. Define features (X) and target (y)
X_train = train_df.drop(columns=["new_deaths_smoothed_per_million", "country", "week"])
y_train = train_df["new_deaths_smoothed_per_million"]

X_test = test_df.drop(columns=["new_deaths_smoothed_per_million", "country", "week"])
y_test = test_df["new_deaths_smoothed_per_million"]

# 4. Initialize Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# 5. Train model
rf_model.fit(X_train, y_train)

# 6. Predictions
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# 7. Evaluate performance
print("Train R²:", r2_score(y_train, y_pred_train))
print("Test R²:", r2_score(y_test, y_pred_test))

# 8. Feature Importance
rf_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nFeature Importance:")
print(rf_importance)

# 9. (Optional) Plot feature importance
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(rf_importance["Feature"], rf_importance["Importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.show()

---

## Part 4 — Visualizations

*Source: `Visualizations.ipynb`*


In [ ]:
owid = pd.read_csv('../data/owid-covid-data.csv')
display(owid.head())

In [ ]:
owid['stringency_index']

### Visualization #1

In [ ]:
import pycountry
import altair as alt
from vega_datasets import data as vega_data

deaths = (
    owid.groupby(['iso_code', 'location'])['total_deaths_per_million']
    .max()
    .reset_index()
)

# Remove zeros for log scale
deaths = deaths[deaths['total_deaths_per_million'] > 0]

# Convert ISO alpha-3 to numeric ID
def to_numeric(iso3):
    try:
        return int(pycountry.countries.get(alpha_3=iso3).numeric)
    except:
        return None

deaths['id'] = deaths['iso_code'].apply(to_numeric)
deaths = deaths.dropna(subset=['id'])
deaths['id'] = deaths['id'].astype(int)

world = alt.topo_feature(vega_data.world_110m.url, 'countries')

chart = alt.Chart(world).mark_geoshape().encode(
    color=alt.Color(
        'total_deaths_per_million:Q',
        scale=alt.Scale(
            scheme='redyellowgreen',
            reverse=True,
            type='log'
        ),
        legend=alt.Legend(
            title="Deaths per Million (log scale)",
            format=".0f"
        )
    ),
    tooltip=[
        alt.Tooltip('location:N', title='Country'),
        alt.Tooltip('total_deaths_per_million:Q', title='Deaths per Million', format='.1f')
    ]
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(
        data=deaths,
        key='id',
        fields=['total_deaths_per_million', 'location']
    )
).project('naturalEarth1').properties(
    width=800,
    height=450,
    title='COVID-19 Deaths per Million by Country'
)

chart

### Visualization #2

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# --- Prep data ---
owid['date'] = pd.to_datetime(owid['date'])

stringency_monthly = (
    owid[
        (owid['date'] >= '2020-01-01') &
        (owid['date'] <= '2022-12-31') &
        owid['stringency_index'].notna()
    ]
    .groupby(['location', pd.Grouper(key='date', freq='MS')])['stringency_index']
    .mean()
    .reset_index()
)

highlight = ['United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany']

# --- Background lines ---
grey = alt.Chart(
    stringency_monthly[~stringency_monthly['location'].isin(highlight)]
).mark_line(
    opacity=0.08,
    color='grey',
    strokeWidth=0.8
).encode(
    x=alt.X('date:T', title='', axis=alt.Axis(format='%b %Y', tickCount={'interval': 'month', 'step': 3})),
    y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0, 100])),
    detail='location:N'
)

# --- Color scale ---
color_scale = alt.Scale(
    domain=highlight,
    range=['#1f77b4', '#d62728', '#2ca02c', '#8c564b', '#ff7f0e', '#9467bd']
)

# --- Legend interaction ---
highlight_selection = alt.selection_point(fields=['location'], bind='legend')

# --- Highlighted lines ---
lines = alt.Chart(
    stringency_monthly[stringency_monthly['location'].isin(highlight)]
).mark_line(
    strokeWidth=2.5,
    interpolate='monotone'
).encode(
    x='date:T',
    y='stringency_index:Q',
    color=alt.Color('location:N', scale=color_scale, title='Country'),
    opacity=alt.condition(highlight_selection, alt.value(1), alt.value(0.2)),
    tooltip=[
        alt.Tooltip('location:N', title='Country'),
        alt.Tooltip('date:T', title='Date', format='%b %Y'),
        alt.Tooltip('stringency_index:Q', title='Stringency Index', format='.1f')
    ]
).add_params(highlight_selection)

# --- Event annotations ---
events = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',       'y': 95},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',         'y': 85},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',          'y': 75},
    {'date': '2020-06-08', 'label': 'NZ Reopens',           'y': 95},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',      'y': 85},
    {'date': '2021-07-01', 'label': 'Delta Wave',           'y': 75},
    {'date': '2021-12-01', 'label': 'Omicron Wave',         'y': 95},
    {'date': '2022-03-01', 'label': 'Restrictions Lifted',  'y': 85},
])
events['date'] = pd.to_datetime(events['date'])

annotation_rules = alt.Chart(events).mark_rule(
    strokeDash=[4, 4],
    opacity=0.4,
    color='black'
).encode(
    x='date:T'
)

annotation_text = alt.Chart(events).mark_text(
    align='left',
    dx=5,
    dy=-5,
    fontSize=9,
    color='#333333',
    fontStyle='italic'
).encode(
    x='date:T',
    y='y:Q',
    text='label:N'
)

# --- Wave shading ---
wave_regions = pd.DataFrame([
    {'start': '2021-06-01', 'end': '2021-10-01'},
    {'start': '2021-11-15', 'end': '2022-02-15'}
])
wave_regions['start'] = pd.to_datetime(wave_regions['start'])
wave_regions['end'] = pd.to_datetime(wave_regions['end'])

bands = alt.Chart(wave_regions).mark_rect(
    opacity=0.08,
    color='gray'
).encode(
    x='start:T',
    x2='end:T'
)

# --- Combine everything ---
chart = (
    bands +
    grey +
    lines +
    annotation_rules +
    annotation_text
).properties(
    width=850,
    height=480,
    title=alt.TitleParams(
        'COVID-19 Oxford Stringency Index by Country (2020\u20132022)',
        fontSize=15
    )
).configure_axis(
    grid=True,
    gridOpacity=0.2
).configure_view(
    strokeWidth=0
)

chart

### Visualization #? Mobility Data

In [ ]:
mobility_df = pd.read_csv("../data/mobility_report_countries.csv")
mobility_df = mobility_df[mobility_df["region"] == "Total"]
mobility_df.reset_index()

mobility_df["date"] = pd.to_datetime(mobility_df["date"])
out_cols = [
    'retail and recreation',
    'grocery and pharmacy',
    'parks',
    'transit stations',
    'workplaces'
]
mobility_df['total_out'] = mobility_df[out_cols].mean(axis=1) 

mobility_df['week'] = mobility_df['date'].dt.to_period('W')

wmobility_df = mobility_df.groupby(['week',"country"]).agg({
    'total_out': 'mean',
    'residential': 'mean'
}).reset_index()

wmobility_df.head()

s = pd.read_csv("../data/OxCGRT_compact_national_v1.csv")
s['Date'] = pd.to_datetime(s['Date'], format='%Y%m%d')
strin = s.copy()
strin['week'] = strin['Date'].dt.to_period('W')
weekly_stringency = strin.groupby(['week', "CountryName"])['StringencyIndex_Average'].mean().reset_index()


In [ ]:
alt.data_transformers.enable("vegafusion")

# Event dates
events = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',       'y': 75},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',         'y': 68},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',          'y': 56},
    {'date': '2020-06-08', 'label': 'NZ Reopens',           'y': 70},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',      'y': 72},
    {'date': '2021-07-01', 'label': 'Delta Wave',           'y': 66},
    {'date': '2021-12-01', 'label': 'Omicron Wave',         'y': 68},
    {'date': '2022-03-01', 'label': 'Restrictions Lifted',  'y': 72},
])

countries = [
    "United States",
    "New Zealand",
    "Sweden",
    "Brazil",
    "Germany"
]

# Use ALL countries, not just the highlighted ones
plot_df = wmobility_df.copy()

# Make sure week is datetime
if str(plot_df["week"].dtype).startswith("period"):
    plot_df["week"] = plot_df["week"].dt.start_time
else:
    plot_df["week"] = pd.to_datetime(plot_df["week"])

events["date"] = pd.to_datetime(events["date"])

# Flag highlighted countries
plot_df["highlight"] = plot_df["country"].isin(countries)

# Optional: fixed x-axis so the plot does not rescale
x_min = plot_df["week"].min()
x_max = plot_df["week"].max()

# Smoothed chart source
smooth_chart = alt.Chart(plot_df).transform_window(
    smoothed_residential="mean(residential)",
    frame=[-2, 2],
    groupby=["country"]
)

# Background lines for all countries
lines = smooth_chart.mark_line().encode(
    x=alt.X(
        "week:T",
        title="Week",
        scale=alt.Scale(domain=[x_min, x_max])
    ),
    y=alt.Y(
        "smoothed_residential:Q",
        title="Residential Mobility (% Change from Baseline)",
        scale = alt.Scale(domain = [-15,80])
    ),
    detail="country:N",
    color=alt.condition(
        alt.datum.highlight,
        alt.Color(
            "country:N",
            title="Highlighted Countries",
            scale=alt.Scale(
                domain=countries,
                range=["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]
            )
        ),
        alt.value("lightgray")
    ),
    size=alt.condition(alt.datum.highlight, alt.value(2.5), alt.value(1)),
    opacity=alt.condition(alt.datum.highlight, alt.value(1.0), alt.value(0.5)),
    tooltip=[
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("smoothed_residential:Q", title="Smoothed Residential", format=".2f")
    ]
)

# Invisible points for cleaner tooltip interaction
points = smooth_chart.mark_circle(size=50, opacity=0).encode(
    x="week:T",
    y="smoothed_residential:Q",
    detail="country:N",
    tooltip=[
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("smoothed_residential:Q", title="Smoothed Residential", format=".2f")
    ]
)

# Vertical event lines
event_rules = alt.Chart(events).mark_rule(strokeDash=[4, 4], size=1).encode(
    x="date:T"
)

# Event labels
event_text = alt.Chart(events).mark_text(
    angle=0,
    align="left",
    dx=5,
    dy=-5,
    fontSize=11
).encode(
    x="date:T",
    y=alt.Y("y:Q"),
    text="label:N"
)

chart = (lines + points + event_rules + event_text).properties(
    title="Smoothed Residential Mobility Over Time Across Countries",
    width=850,
    height=450
)

chart

In [ ]:
weekly_stringency = weekly_stringency.rename(columns={'CountryName': 'country'})
merged_df = pd.merge(
    weekly_stringency,
    wmobility_df,
    on=['country', 'week'],
    how='inner'  # or 'left' depending on your goal
)
# Convert week to datetime
if str(merged_df["week"].dtype).startswith("period"):
    merged_df["week"] = merged_df["week"].dt.to_timestamp()
else:
    merged_df["week"] = pd.to_datetime(merged_df["week"])

# Clean data
plot_df = merged_df.dropna(subset=["country", "week", "residential", "StringencyIndex_Average"]).copy()
plot_df["week"] = pd.to_datetime(plot_df["week"])

# Compute correlation for each country
corr_df = (
    plot_df.groupby("country")
    .apply(lambda x: x["StringencyIndex_Average"].corr(x["residential"]))
    .reset_index(name="corr_value")
)

# Merge correlation back into plotting df
plot_df = plot_df.merge(corr_df, on="country", how="left")

# Dropdown list of countries
country_list = sorted(plot_df["country"].unique())

country_select = alt.param(
    name="SelectedCountry",
    bind=alt.binding_select(options=country_list, name="Choose a country: "),
    value=country_list[0]
)

# Base chart filtered by selected country
base = alt.Chart(plot_df).transform_filter(
    "datum.country == SelectedCountry"
)

# Scatter plot
points = base.mark_circle(size=70, opacity=0.6).encode(
    x=alt.X("StringencyIndex_Average:Q", title="Stringency Index"),
    y=alt.Y("residential:Q", title="Residential Mobility (% Change from Baseline)"),
    tooltip=[
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("week:T", title="Week"),
        alt.Tooltip("StringencyIndex_Average:Q", title="Stringency", format=".2f"),
        alt.Tooltip("residential:Q", title="Residential Mobility", format=".2f"),
    ]
)

# Regression line
trend = base.transform_regression(
    "StringencyIndex_Average",
    "residential"
).mark_line(size=3, color="black").encode(
    x="StringencyIndex_Average:Q",
    y="residential:Q"
)

# R² text
r2_text = base.transform_regression(
    "StringencyIndex_Average",
    "residential",
    params=True
).transform_calculate(
    label="'R² = ' + format(datum.rSquared, '.2f')"
).mark_text(
    align="left",
    baseline="top",
    fontSize=13
).encode(
    x=alt.value(10),
    y=alt.value(10),
    text="label:N"
)

# Correlation text
corr_text = base.transform_aggregate(
    corr_value="max(corr_value)"
).transform_calculate(
    label="'Correlation = ' + format(datum.corr_value, '.2f')"
).mark_text(
    align="left",
    baseline="top",
    fontSize=13
).encode(
    x=alt.value(10),
    y=alt.value(30),
    text="label:N"
)

# Final interactive chart
chart = (points + trend + r2_text + corr_text).add_params(
    country_select
).properties(
    width=500,
    height=400,
    title="Residential Mobility vs Stringency Index by Country"
)

chart

### Visualization #3

In [ ]:
owid = pd.read_csv('../data/owid-covid-data.csv')

In [ ]:
# setting date to date time format
owid['date'] = pd.to_datetime(owid['date'])

# First confirmed death per country 
first_death = (
    owid[owid['total_deaths'] > 0]
    .groupby('location')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'first_death_date'})
)

# First strong restriction date (stringency ≥ 70) 
strong_restriction = (
    owid[owid['stringency_index'] >= 70]
    .groupby('location')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'restriction_date'})
)

# Merge
country_dates = pd.merge(first_death, strong_restriction, on='location', how='inner')

# Calculate days between
country_dates['days_to_restrictions'] = (
    country_dates['restriction_date'] - country_dates['first_death_date']
).dt.days

# Get deaths per million 
deaths_pm = (
    owid.groupby('location')['total_deaths_per_million']
    .max()
    .reset_index()
)

# Get continent + population
meta = (
    owid.groupby('location')
    .agg({
        'continent': 'first',
        'population': 'first'
    })
    .reset_index()
)

# Final dataset 
country_owid = (
    country_dates
    .merge(deaths_pm, on='location')
    .merge(meta, on='location')
)

# Drop missing continent rows 
country_owid = country_owid[country_owid['continent'].notna()]

country_owid.head()

In [ ]:
# remove countries that are past the x-axis limit
country_owid = country_owid[country_owid['days_to_restrictions'] < 200]

# creating death/population percentage variable
country_owid['death_pct'] = (
    country_owid['total_deaths_per_million'] / 1_000_000
) * 100

# countries to label
#label_countries = [
#    "United States", "Italy", "China", "India", "Brazil",
#    "United Kingdom", "Sweden", "New Zealand"
#]

label_countries = [
    "United States",
    "United Kingdom",
    "Canada",
    "China",
    "Germany",
    "Italy",
    "Sweden",
    "Australia",
    "Japan"
]

# color scale for countries
continent_scale = alt.Scale(
    domain=['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania']
)

# brush for selction interval
brush = alt.selection_interval()

# Base scatterplot
points = (
    alt.Chart(country_owid)
    .mark_circle(opacity=0.7, stroke='black', strokeWidth=0.3)
    .encode(
        x=alt.X(
            'days_to_restrictions:Q',
            title='Number of Days Between First COVID Death and Strong Restrictions', 
            scale=alt.Scale(domain=[-750, 150])
        ),
        y=alt.Y(
            'total_deaths_per_million:Q',
            title='COVID Deaths per Million'
        ),
        color=alt.Color(
            'continent:N', 
            scale=continent_scale
        ),
        size=alt.Size(
            'population:Q',
            title='Population',
            scale=alt.Scale(range=[30, 2000])   #  bubble size
        ),
        tooltip=[
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('continent:N', title='Continent'),
            alt.Tooltip('days_to_restrictions:Q', title='Days to Restrictions'),
            alt.Tooltip('total_deaths_per_million:Q', title='Deaths per Million', format='.2f'),
            alt.Tooltip('population:Q', title='Population', format=',')
        ], 
        opacity=alt.condition(brush, alt.value(0.7), alt.value(0.1))
    )
).add_params(brush)


# Labels for selected countries
labels_df = country_owid[country_owid['location'].isin(label_countries)]

labels = (
    alt.Chart(labels_df)
    .mark_text(
        align='left',
        baseline='middle',
        dx=7,
        dy=-7,
        fontSize=11
    )
    .encode(
        x='days_to_restrictions:Q',
        y='total_deaths_per_million:Q',
        text='location:N',
        color=alt.value('#333333')
    )
)

zero_line = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(
    color='gray',
    strokeDash=[4, 4]
).encode(
    x='x:Q'
)

zero_label = alt.Chart(
    pd.DataFrame({
        'x': [-200],
        'y': [6000], 
        'label': ['Restrictions implemented near first death']
    })
).mark_text(
    align='left',
    dx=7
).encode(
    x='x:Q',
    y='y:Q',
    text='label:N'
)

# histogram of the total deaths
hist = (
    alt.Chart(country_owid)
    .mark_bar()
    .encode(
        y=alt.Y(
            'death_pct:Q',
            bin=alt.Bin(maxbins=30),
            title='COVID Deaths (& of Population)'
        ),
        x=alt.X(
            'count():Q',
            title='Number of Countries'
        ), 
        color=alt.Color(
            'continent:N', 
            scale=continent_scale
        )
    )
    .transform_filter(brush)
)

# Combine 

points = points.properties(
    width=800, 
    height=500, 
    title='Policy Timing vs COVID Deaths'
)

hist = hist.properties(
    width = 500, 
    height= 200, 
    title='COVID Deaths (& of Population) Distribution'
)

chart = (points + labels + zero_line + zero_label) & hist

chart = chart.configure_axis(
    labelFontSize=12,
    titleFontSize=13
).configure_legend(
    titleFontSize=12,
    labelFontSize=11
)


chart 

#### Visualization #4

In [ ]:
owid = pd.read_csv('../data/owid-covid-data.csv')

In [ ]:
# Prep data
owid['date'] = pd.to_datetime(owid['date'])

map_df = owid[
    owid['continent'].notna() &
    owid['iso_code'].notna() &
    owid['stringency_index'].notna()
].copy()

# Remove OWID aggregates like World, income groups, etc.
map_df = map_df[~map_df['iso_code'].str.startswith('OWID')]

# Convert to month
map_df['month'] = map_df['date'].dt.to_period('M').dt.to_timestamp()
map_df['month_label'] = map_df['month'].dt.strftime('%Y-%m')

# Monthly average stringency by country
monthly_stringency = (
    map_df.groupby(['iso_code', 'location', 'month_label'], as_index=False)['stringency_index']
    .mean()
)

# Plotly choropleth
fig = px.choropleth(
    monthly_stringency,
    locations='iso_code',
    color='stringency_index',
    hover_name='location',
    animation_frame='month_label',
    color_continuous_scale='RdYlGn',
    range_color=(0, 100),
    projection='natural earth',
    labels={'stringency_index': 'Stringency Index'},
    title='Monthly COVID Stringency Index by Country'
)

fig.update_layout(
    width=1000,
    height=600,
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

#### Visualization #4.2

In [ ]:
alt.data_transformers.disable_max_rows()
# -----------------------------
# 1) Choose countries
# -----------------------------
peer_countries = [
    "United States",
    "United Kingdom",
    "Canada",
    "China",
    "Germany",
    "Italy",
    "Sweden",
    "Australia",
    "Japan"
]

# -----------------------------
# 2) Prep data
# -----------------------------
owid['date'] = pd.to_datetime(owid['date'])

ts_df = owid[
    owid['location'].isin(peer_countries)
][[
    'date',
    'location',
    'stringency_index',
    'new_deaths_per_million'
]].copy()

ts_df = ts_df.rename(columns={
    'new_deaths_per_million': 'mortality_rate'
})

# Drop rows where both metrics are missing
ts_df = ts_df[
    ts_df['stringency_index'].notna() | ts_df['mortality_rate'].notna()
]

# -----------------------------
# 3) Interactive country selection
# -----------------------------
selection = alt.selection_point(
    fields=['location'],
    on='click',          
    clear='dblclick'     
)

color = alt.Color(
    'location:N',
    title='Country'
)

# --- Legend interaction ---
highlight_selection = alt.selection_point(fields=['location'], bind='legend')

# -----------------------------
# 4) Stringency chart
# -----------------------------
stringency_chart = (
    alt.Chart(ts_df)
    .mark_line(size=2)
    .encode(
        x=alt.X('date:T', title='Date'),
        y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0, 100])),
        color=color,
        opacity=alt.condition(selection, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip('date:T', title='Date'),
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('stringency_index:Q', title='Stringency Index', format='.1f')
        ]
    )
    .add_params(selection)
    .properties(
        width=850,
        height=280,
        title='COVID Policy Stringency Over Time'
    )
)

# -----------------------------
# 5) Mortality chart
# -----------------------------
mortality_chart = (
    alt.Chart(ts_df)
    .transform_filter(selection) 
    .mark_line(size=2)
    .encode(
        x=alt.X('date:T', title='Date'),
        y=alt.Y(
            'mortality_rate:Q',
            title='New Deaths per Million', 
            scale=alt.Scale(domain=[0, 150]) 
        ),
        color=color,
        opacity=alt.condition(selection, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip('date:T', title='Date'),
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('mortality_rate:Q', title='New Deaths per Million', format='.2f')
        ]
    )
    .properties(
        width=850,
        height=280,
        title='COVID Mortality Trends Over Time'
    )
)

# -----------------------------
# 6) Combine charts
# -----------------------------
final_chart = alt.vconcat(
    stringency_chart,
    mortality_chart
).resolve_scale(
    color='shared'
).properties(
    title='Comparing COVID Policy and Mortality Trends Across Peer Countries'
)

final_chart

---

## Part 5 — Dashboard

*Source: `dashboard.ipynb`*


In [ ]:
%%html
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

body, .jp-Notebook { background: #f0f2f5 !important; font-family: 'Inter', sans-serif !important; }

.hero {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    padding: 2.5rem 2rem;
    border-radius: 16px;
    margin-bottom: 1.5rem;
    color: white;
}
.hero h1 { font-size: 2rem; font-weight: 700; margin: 0 0 0.4rem 0; letter-spacing: -0.5px; }
.hero p  { font-size: 0.95rem; opacity: 0.75; margin: 0; font-weight: 300; }
.hero-badge {
    display: inline-block;
    background: rgba(255,255,255,0.15);
    border: 1px solid rgba(255,255,255,0.25);
    border-radius: 20px;
    padding: 3px 14px;
    font-size: 0.75rem;
    font-weight: 500;
    margin-bottom: 0.8rem;
    letter-spacing: 0.5px;
}

.kpi-row { display: flex; gap: 1rem; margin-bottom: 1.5rem; flex-wrap: wrap; }
.kpi-card {
    flex: 1; min-width: 180px;
    background: white;
    border: 1px solid #e8ecf0;
    border-radius: 12px;
    padding: 1.2rem 1.4rem;
    box-shadow: 0 1px 6px rgba(0,0,0,0.06);
    position: relative;
}
.kpi-label { font-size: 0.7rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.8px; color: #8492a6; margin-bottom: 0.3rem; }
.kpi-value { font-size: 1.8rem; font-weight: 700; color: #1a1a2e; line-height: 1; margin-bottom: 0.2rem; }
.kpi-sub   { font-size: 0.75rem; color: #8492a6; }
.kpi-icon  { font-size: 2rem; position: absolute; top: 1rem; right: 1.2rem; opacity: 0.12; }

.section-header { font-size: 1rem; font-weight: 600; color: #1a1a2e; margin: 0.5rem 0 0.2rem 0; }
.section-desc   { font-size: 0.82rem; color: #8492a6; margin-bottom: 0.8rem; }

.fancy-divider { border: none; height: 1px; background: linear-gradient(to right, transparent, #d8dce4, transparent); margin: 1.2rem 0; }

.widget-tab > .p-TabBar { background: #eef0f4; border-radius: 10px 10px 0 0; padding: 4px 6px 0; }
.widget-tab > .p-TabBar .p-TabBar-tab { font-family: 'Inter', sans-serif !important; font-size: 0.85rem !important; font-weight: 500; border-radius: 8px 8px 0 0; }
.widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current { font-weight: 600 !important; background: white !important; }
.widget-tab > .widget-tab-contents { background: white; border-radius: 0 0 12px 12px; border: 1px solid #e8ecf0; padding: 1.2rem; box-shadow: 0 2px 8px rgba(0,0,0,0.05); }
</style>

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import plotly.express as px
import pycountry
import ipywidgets as widgets
from IPython.display import display, HTML
from vega_datasets import data as vega_data
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

alt.data_transformers.enable('vegafusion')

# ── OWID ──────────────────────────────────────────────────────────────────────
owid = pd.read_csv('../data/owid-covid-data.csv')
owid['date'] = pd.to_datetime(owid['date'])

# ── Google Mobility ───────────────────────────────────────────────────────────
mob_raw = pd.read_csv('../data/mobility_report_countries.csv')
mob_raw = mob_raw[mob_raw['region'] == 'Total']
mob_raw['date'] = pd.to_datetime(mob_raw['date'])
out_cols = ['retail and recreation', 'grocery and pharmacy', 'parks', 'transit stations', 'workplaces']
mob_raw['total_out'] = mob_raw[out_cols].mean(axis=1)
mob_raw['week'] = mob_raw['date'].dt.to_period('W')
wmobility_df = (
    mob_raw.groupby(['week', 'country'])
    .agg(total_out=('total_out', 'mean'), residential=('residential', 'mean'))
    .reset_index()
)

# ── OxCGRT ────────────────────────────────────────────────────────────────────
oxcgrt = pd.read_csv('../data/OxCGRT_compact_national_v1.csv')
oxcgrt['Date'] = pd.to_datetime(oxcgrt['Date'], format='%Y%m%d')
oxcgrt['week'] = oxcgrt['Date'].dt.to_period('W')
weekly_stringency = (
    oxcgrt.groupby(['week', 'CountryName'])['StringencyIndex_Average']
    .mean().reset_index()
    .rename(columns={'CountryName': 'country'})
)

# ── ML feature data ───────────────────────────────────────────────────────────
selected_countries = [
    'United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany',
    'United Kingdom', 'France', 'Italy', 'Spain', 'Canada', 'Australia',
    'Japan', 'South Korea', 'Singapore', 'Netherlands', 'Belgium', 'Denmark', 'Norway'
]
owid_ml = owid[owid['location'].isin(selected_countries)].copy()
owid_ml = owid_ml[~owid_ml['iso_code'].str.startswith('OWID')]
owid_ml = owid_ml.rename(columns={'location': 'country'})
owid_ml['week'] = owid_ml['date'].dt.to_period('W').dt.start_time
owid_ml = owid_ml.sort_values(['country', 'date'])

avg_cols  = [c for c in ['new_deaths_smoothed_per_million', 'new_cases_smoothed_per_million',
                          'stringency_index', 'reproduction_rate'] if c in owid_ml.columns]
last_cols = [c for c in ['gdp_per_capita', 'median_age', 'hospital_beds_per_thousand',
                          'population_density', 'aged_65_older', 'human_development_index',
                          'diabetes_prevalence', 'life_expectancy',
                          'people_vaccinated_per_hundred'] if c in owid_ml.columns]

agg_dict = {**{c: 'mean' for c in avg_cols}, **{c: 'last' for c in last_cols}}
weekly_ml = owid_ml.groupby(['country', 'week']).agg(agg_dict).reset_index()
weekly_ml = weekly_ml.sort_values(['country', 'week'])
weekly_ml[avg_cols] = weekly_ml.groupby('country')[avg_cols].ffill()
if 'people_vaccinated_per_hundred' in weekly_ml.columns:
    weekly_ml['people_vaccinated_per_hundred'] = weekly_ml['people_vaccinated_per_hundred'].fillna(0)

weekly_ml['stringency_lag_2'] = weekly_ml.groupby('country')['stringency_index'].shift(2)
weekly_ml = weekly_ml.dropna(subset=['new_deaths_smoothed_per_million'])

all_features = ['stringency_lag_2', 'new_cases_smoothed_per_million', 'aged_65_older',
                'human_development_index', 'population_density', 'reproduction_rate',
                'diabetes_prevalence', 'hospital_beds_per_thousand',
                'people_vaccinated_per_hundred', 'life_expectancy']
features = [f for f in all_features if f in weekly_ml.columns]
df_model = weekly_ml[features + ['new_deaths_smoothed_per_million', 'country', 'week']].dropna()

In [ ]:
countries_tracked = owid[
    owid['continent'].notna() & ~owid['iso_code'].str.startswith('OWID', na=True)
]['location'].nunique()

total_deaths = owid.groupby('location')['total_deaths'].max().sum()
peak_stringency = owid['stringency_index'].max()
date_range = f"{owid['date'].min().strftime('%b %Y')} \u2013 {owid['date'].max().strftime('%b %Y')}"

display(HTML(f'''
<div class="hero">
    <div class="hero-badge">\U0001f393 CS 329E \u00b7 University of Texas at Austin</div>
    <h1>\U0001f9a0 COVID-19 Policy &amp; Mortality Dashboard</h1>
    <p>Exploring how government responses shaped pandemic outcomes across 200+ countries &amp; territories</p>
</div>
<div class="kpi-row">
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f30d</div>
        <div class="kpi-label">Countries Tracked</div>
        <div class="kpi-value">{countries_tracked}</div>
        <div class="kpi-sub">across 6 continents</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f4ca</div>
        <div class="kpi-label">Confirmed Deaths</div>
        <div class="kpi-value">{total_deaths/1e6:.1f}M</div>
        <div class="kpi-sub">cumulative worldwide</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f4cb</div>
        <div class="kpi-label">Peak Stringency</div>
        <div class="kpi-value">{peak_stringency:.0f}</div>
        <div class="kpi-sub">out of 100 (Oxford Index)</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-icon">\U0001f4c5</div>
        <div class="kpi-label">Date Range</div>
        <div class="kpi-value" style="font-size:1.2rem">{date_range}</div>
        <div class="kpi-sub">full pandemic timeline</div>
    </div>
</div>
<hr class="fancy-divider">
'''))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHARTS 1–5
# ══════════════════════════════════════════════════════════════════════════════

# ── Chart 1: Deaths per million choropleth ────────────────────────────────────
def to_numeric(iso3):
    try:
        return int(pycountry.countries.get(alpha_3=iso3).numeric)
    except Exception:
        return None

deaths = owid.groupby(['iso_code', 'location'])['total_deaths_per_million'].max().reset_index()
deaths = deaths[deaths['total_deaths_per_million'] > 0]
deaths['id'] = deaths['iso_code'].apply(to_numeric)
deaths = deaths.dropna(subset=['id'])
deaths['id'] = deaths['id'].astype(int)

world = alt.topo_feature(vega_data.world_110m.url, 'countries')

chart1 = (
    alt.Chart(world).mark_geoshape(stroke='white', strokeWidth=0.4)
    .encode(
        color=alt.Color('total_deaths_per_million:Q',
                        scale=alt.Scale(scheme='redyellowgreen', reverse=True, type='log'),
                        legend=alt.Legend(title='Deaths / Million (log)', format='.0f', orient='bottom-right')),
        tooltip=[alt.Tooltip('location:N', title='Country'),
                 alt.Tooltip('total_deaths_per_million:Q', title='Deaths per Million', format=',.0f')],
    )
    .transform_lookup(lookup='id',
                      from_=alt.LookupData(data=deaths, key='id',
                                           fields=['total_deaths_per_million', 'location']))
    .project('naturalEarth1')
    .properties(width=900, height=480)
    .configure_view(strokeWidth=0)
)

# ── Chart 2: Stringency over time ─────────────────────────────────────────────
stringency_monthly = (
    owid[(owid['date'] >= '2020-01-01') & (owid['date'] <= '2022-12-31')
         & owid['stringency_index'].notna()]
    .groupby(['location', pd.Grouper(key='date', freq='MS')])['stringency_index']
    .mean().reset_index()
)

highlight2 = ['United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany']
color_scale2 = alt.Scale(domain=highlight2,
                         range=['#1f77b4', '#d62728', '#2ca02c', '#8c564b', '#ff7f0e', '#9467bd'])

grey2 = (
    alt.Chart(stringency_monthly[~stringency_monthly['location'].isin(highlight2)])
    .mark_line(opacity=0.07, color='#aab4c4', strokeWidth=0.7)
    .encode(x=alt.X('date:T', title='',
                     axis=alt.Axis(format='%b %Y', tickCount={'interval': 'month', 'step': 3})),
            y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0, 100])),
            detail='location:N')
)

highlight_sel2 = alt.selection_point(fields=['location'], bind='legend')
lines2 = (
    alt.Chart(stringency_monthly[stringency_monthly['location'].isin(highlight2)])
    .mark_line(strokeWidth=2.5, interpolate='monotone')
    .encode(x='date:T', y='stringency_index:Q',
            color=alt.Color('location:N', scale=color_scale2, title='Country'),
            opacity=alt.condition(highlight_sel2, alt.value(1), alt.value(0.15)),
            tooltip=[alt.Tooltip('location:N', title='Country'),
                     alt.Tooltip('date:T', title='Date', format='%b %Y'),
                     alt.Tooltip('stringency_index:Q', title='Stringency Index', format='.1f')])
    .add_params(highlight_sel2)
)

events2 = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',      'y': 95},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',        'y': 85},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',         'y': 75},
    {'date': '2020-06-08', 'label': 'NZ Reopens',          'y': 95},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',     'y': 85},
    {'date': '2021-07-01', 'label': 'Delta Wave',          'y': 75},
    {'date': '2021-12-01', 'label': 'Omicron Wave',        'y': 95},
    {'date': '2022-03-01', 'label': 'Restrictions Lifted', 'y': 85},
])
events2['date'] = pd.to_datetime(events2['date'])
ann_rules2 = alt.Chart(events2).mark_rule(strokeDash=[4, 4], opacity=0.35, color='#555').encode(x='date:T')
ann_text2  = (alt.Chart(events2)
              .mark_text(align='left', dx=5, dy=-5, fontSize=9, color='#555', fontStyle='italic')
              .encode(x='date:T', y='y:Q', text='label:N'))

waves2 = pd.DataFrame([{'start': '2021-06-01', 'end': '2021-10-01'},
                        {'start': '2021-11-15', 'end': '2022-02-15'}])
waves2['start'] = pd.to_datetime(waves2['start'])
waves2['end']   = pd.to_datetime(waves2['end'])
bands2 = alt.Chart(waves2).mark_rect(opacity=0.06, color='#0f3460').encode(x='start:T', x2='end:T')

chart2 = (
    (bands2 + grey2 + lines2 + ann_rules2 + ann_text2)
    .properties(width=900, height=460)
    .configure_axis(grid=True, gridOpacity=0.15, labelFontSize=11, titleFontSize=12)
    .configure_view(strokeWidth=0)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

# ── Chart 3: Policy timing vs mortality (brush + linked histogram) ────────────────────────────────
first_death3 = (owid[owid['total_deaths'] > 0].groupby('location')['date']
                .min().reset_index().rename(columns={'date': 'first_death_date'}))
strong_restr3 = (owid[owid['stringency_index'] >= 70].groupby('location')['date']
                 .min().reset_index().rename(columns={'date': 'restriction_date'}))
country_dates3 = pd.merge(first_death3, strong_restr3, on='location', how='inner')
country_dates3['days_to_restrictions'] = (
    country_dates3['restriction_date'] - country_dates3['first_death_date']).dt.days

deaths_pm3 = owid.groupby('location')['total_deaths_per_million'].max().reset_index()
meta3 = owid.groupby('location').agg(continent=('continent', 'first'),
                                     population=('population', 'first')).reset_index()
country_owid3 = country_dates3.merge(deaths_pm3, on='location').merge(meta3, on='location')
country_owid3 = country_owid3[country_owid3['continent'].notna()]
country_owid3 = country_owid3[country_owid3['days_to_restrictions'] < 200]
country_owid3['death_pct'] = (country_owid3['total_deaths_per_million'] / 1_000_000) * 100

label_countries3 = ['United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany']
continent_scale3 = alt.Scale(
    domain=['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania'])
brush3 = alt.selection_interval()

points3 = (
    alt.Chart(country_owid3).mark_circle(opacity=0.7, stroke='black', strokeWidth=0.3)
    .encode(
        x=alt.X('days_to_restrictions:Q',
                 title='Number of Days Between First COVID Death and Strong Restrictions',
                 scale=alt.Scale(domain=[-750, 150])),
        y=alt.Y('total_deaths_per_million:Q', title='COVID Deaths per Million'),
        color=alt.Color('continent:N', scale=continent_scale3),
        size=alt.Size('population:Q', title='Population', scale=alt.Scale(range=[30, 2000])),
        opacity=alt.condition(brush3, alt.value(0.7), alt.value(0.1)),
        tooltip=[alt.Tooltip('location:N', title='Country'),
                 alt.Tooltip('continent:N', title='Continent'),
                 alt.Tooltip('days_to_restrictions:Q', title='Days to Restrictions'),
                 alt.Tooltip('total_deaths_per_million:Q', title='Deaths per Million', format='.2f'),
                 alt.Tooltip('population:Q', title='Population', format=',')]
    )
    .add_params(brush3)
    .properties(width=800, height=500, title='Policy Timing vs COVID Deaths')
)
labels3 = (
    alt.Chart(country_owid3[country_owid3['location'].isin(label_countries3)])
    .mark_text(align='left', baseline='middle', dx=7, dy=-7, fontSize=11)
    .encode(x='days_to_restrictions:Q', y='total_deaths_per_million:Q',
            text='location:N', color=alt.value('#333333'))
)
zero_line3  = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='gray', strokeDash=[4, 4]).encode(x='x:Q')
zero_label3 = (
    alt.Chart(pd.DataFrame({'x': [-200], 'y': [6000],
                            'label': ['Restrictions implemented near first death']}))
    .mark_text(align='left', dx=7).encode(x='x:Q', y='y:Q', text='label:N')
)
hist3 = (
    alt.Chart(country_owid3).mark_bar()
    .encode(
        y=alt.Y('death_pct:Q', bin=alt.Bin(maxbins=30), title='COVID Deaths (% of Population)'),
        x=alt.X('count():Q', title='Number of Countries'),
        color=alt.Color('continent:N', scale=continent_scale3)
    )
    .transform_filter(brush3)
    .properties(width=500, height=200, title='COVID Deaths (% of Population) Distribution')
)
chart3 = (
    ((points3 + labels3 + zero_line3 + zero_label3) & hist3)
    .configure_axis(labelFontSize=12, titleFontSize=13)
    .configure_legend(titleFontSize=12, labelFontSize=11)
)

# ── Chart 4: Animated Plotly choropleth ───────────────────────────────────────
map_df4 = owid[owid['continent'].notna() & owid['iso_code'].notna()
               & owid['stringency_index'].notna()].copy()
map_df4 = map_df4[~map_df4['iso_code'].str.startswith('OWID')]
map_df4['month_label'] = map_df4['date'].dt.to_period('M').dt.to_timestamp().dt.strftime('%Y-%m')
monthly_s4 = map_df4.groupby(['iso_code', 'location', 'month_label'],
                              as_index=False)['stringency_index'].mean()

fig4 = px.choropleth(monthly_s4, locations='iso_code', color='stringency_index',
                     hover_name='location', animation_frame='month_label',
                     color_continuous_scale='RdYlGn', range_color=(0, 100),
                     projection='natural earth',
                     labels={'stringency_index': 'Stringency Index'})
fig4.update_layout(
    height=520, margin=dict(l=0, r=0, t=10, b=0),
    paper_bgcolor='rgba(0,0,0,0)',
    geo=dict(bgcolor='rgba(0,0,0,0)', showframe=False,
             showcoastlines=True, coastlinecolor='#ccc'),
    coloraxis_colorbar=dict(title='Stringency', thickness=14, len=0.6),
    font=dict(family='Inter, sans-serif'),
)

# ── Chart 5: Policy & mortality trends ────────────────────────────────────────
peer5 = ['United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany']
ts5 = (owid[owid['location'].isin(peer5)]
       [['date', 'location', 'stringency_index', 'new_deaths_per_million']].copy()
       .rename(columns={'new_deaths_per_million': 'mortality_rate'}))
ts5 = ts5[ts5['stringency_index'].notna() | ts5['mortality_rate'].notna()]
ts5['week'] = ts5['date'].dt.to_period('W').dt.start_time
weekly5 = (ts5.groupby(['week', 'location'], as_index=False)
           .agg(stringency_index=('stringency_index', 'mean'),
                mortality_rate=('mortality_rate', 'mean')))

sel5   = alt.selection_point(fields=['location'], on='click', clear='dblclick', bind='legend')
color5 = alt.Color('location:N', title='Country', scale=alt.Scale(scheme='tableau10'))

s_chart5 = (
    alt.Chart(weekly5).mark_line(size=2, interpolate='monotone')
    .encode(x=alt.X('week:T', title=''),
            y=alt.Y('stringency_index:Q', title='Stringency Index',
                    scale=alt.Scale(domain=[0, 100])),
            color=color5,
            opacity=alt.condition(sel5, alt.value(1), alt.value(0.1)),
            tooltip=[alt.Tooltip('week:T', title='Week'),
                     alt.Tooltip('location:N', title='Country'),
                     alt.Tooltip('stringency_index:Q', title='Stringency', format='.1f')])
    .add_params(sel5)
    .properties(width=900, height=250,
                title=alt.TitleParams('Policy Stringency Over Time',
                                      fontSize=13, fontWeight=600))
)
m_chart5 = (
    alt.Chart(weekly5).transform_filter(sel5).mark_line(size=2, interpolate='monotone')
    .encode(x=alt.X('week:T', title='Date'),
            y=alt.Y('mortality_rate:Q', title='New Deaths per Million'),
            color=color5,
            opacity=alt.condition(sel5, alt.value(1), alt.value(0.1)),
            tooltip=[alt.Tooltip('week:T', title='Week'),
                     alt.Tooltip('location:N', title='Country'),
                     alt.Tooltip('mortality_rate:Q', title='Deaths per Million', format='.2f')])
    .properties(width=900, height=250,
                title=alt.TitleParams('Mortality Trends Over Time',
                                      fontSize=13, fontWeight=600))
)
chart5 = (
    alt.vconcat(s_chart5, m_chart5).resolve_scale(color='shared')
    .configure_axis(grid=True, gridOpacity=0.12, labelFontSize=11, titleFontSize=12)
    .configure_view(strokeWidth=0)
    .configure_legend(titleFontSize=12, labelFontSize=11, orient='right')
)

# ══════════════════════════════════════════════════════════════════════════════
# CHARTS 6–9  (new)
# ══════════════════════════════════════════════════════════════════════════════

# ── Chart 6: Residential mobility over time ───────────────────────────────────
highlight_mob = ['United States', 'New Zealand', 'Sweden', 'Brazil', 'Germany']
color_mob = alt.Scale(domain=highlight_mob,
                      range=['#1f77b4', '#2ca02c', '#8c564b', '#ff7f0e', '#9467bd'])

plot_mob = wmobility_df.copy()
if str(plot_mob['week'].dtype).startswith('period'):
    plot_mob['week'] = plot_mob['week'].dt.start_time
else:
    plot_mob['week'] = pd.to_datetime(plot_mob['week'])

mob_grey6 = plot_mob[~plot_mob['country'].isin(highlight_mob)]
mob_hi6   = plot_mob[plot_mob['country'].isin(highlight_mob)]

grey_lines6 = (
    alt.Chart(mob_grey6)
    .transform_window(smoothed_residential='mean(residential)', frame=[-2, 2], groupby=['country'])
    .mark_line(opacity=0.35, color='lightgray', strokeWidth=0.8)
    .encode(x=alt.X('week:T', title='Week'),
            y=alt.Y('smoothed_residential:Q',
                    title='Residential Mobility (% Change from Baseline)',
                    scale=alt.Scale(domain=[-15, 80])),
            detail='country:N')
)
hi_lines6 = (
    alt.Chart(mob_hi6)
    .transform_window(smoothed_residential='mean(residential)', frame=[-2, 2], groupby=['country'])
    .mark_line(strokeWidth=2.5, interpolate='monotone')
    .encode(x='week:T',
            y=alt.Y('smoothed_residential:Q', scale=alt.Scale(domain=[-15, 80])),
            color=alt.Color('country:N', scale=color_mob, title='Country'),
            tooltip=[alt.Tooltip('week:T', title='Week'),
                     alt.Tooltip('country:N', title='Country'),
                     alt.Tooltip('smoothed_residential:Q', title='Residential Mobility', format='.2f')])
)

events6 = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',      'y': 75},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',        'y': 68},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',         'y': 56},
    {'date': '2020-06-08', 'label': 'NZ Reopens',          'y': 70},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',     'y': 72},
    {'date': '2021-07-01', 'label': 'Delta Wave',          'y': 66},
    {'date': '2021-12-01', 'label': 'Omicron Wave',        'y': 68},
    {'date': '2022-03-01', 'label': 'Restrictions Lifted', 'y': 72},
])
events6['date'] = pd.to_datetime(events6['date'])
event_rules6 = (alt.Chart(events6)
                .mark_rule(strokeDash=[4, 4], size=1, opacity=0.4, color='#555').encode(x='date:T'))
event_text6  = (alt.Chart(events6)
                .mark_text(align='left', dx=5, dy=-5, fontSize=9, color='#555', fontStyle='italic')
                .encode(x='date:T', y='y:Q', text='label:N'))

chart6 = (
    (grey_lines6 + hi_lines6 + event_rules6 + event_text6)
    .properties(width=900, height=450,
                title=alt.TitleParams('Smoothed Residential Mobility Over Time Across Countries',
                                      fontSize=13, fontWeight=600))
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridOpacity=0.12, labelFontSize=11, titleFontSize=12)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

# ── Chart 7: Stringency vs residential mobility (per-country dropdown) ────────
merged7 = pd.merge(weekly_stringency, wmobility_df, on=['country', 'week'], how='inner')
if str(merged7['week'].dtype).startswith('period'):
    merged7['week'] = merged7['week'].dt.to_timestamp()
else:
    merged7['week'] = pd.to_datetime(merged7['week'])

plot7 = merged7.dropna(subset=['country', 'week', 'residential', 'StringencyIndex_Average']).copy()

corr7 = (plot7.groupby('country')
         .apply(lambda x: x['StringencyIndex_Average'].corr(x['residential']))
         .reset_index(name='corr_value'))
plot7 = plot7.merge(corr7, on='country', how='left')

country_list7   = sorted(plot7['country'].unique().tolist())
country_select7 = alt.param(
    name='SelectedCountry',
    bind=alt.binding_select(options=country_list7, name='Choose a country: '),
    value=country_list7[0]
)

base7 = alt.Chart(plot7).transform_filter('datum.country == SelectedCountry')

points7 = base7.mark_circle(size=70, opacity=0.6, color='#4c78a8').encode(
    x=alt.X('StringencyIndex_Average:Q', title='Stringency Index'),
    y=alt.Y('residential:Q', title='Residential Mobility (% Change from Baseline)'),
    tooltip=[alt.Tooltip('country:N', title='Country'),
             alt.Tooltip('week:T', title='Week'),
             alt.Tooltip('StringencyIndex_Average:Q', title='Stringency', format='.2f'),
             alt.Tooltip('residential:Q', title='Residential Mobility', format='.2f')]
)
trend7 = (base7.transform_regression('StringencyIndex_Average', 'residential')
          .mark_line(size=3, color='#e45756')
          .encode(x='StringencyIndex_Average:Q', y='residential:Q'))
r2_text7 = (
    base7.transform_regression('StringencyIndex_Average', 'residential', params=True)
    .transform_calculate(label="'R\u00b2 = ' + format(datum.rSquared, '.2f')")
    .mark_text(align='left', baseline='top', fontSize=13, fontWeight=600)
    .encode(x=alt.value(10), y=alt.value(10), text='label:N')
)
corr_text7 = (
    base7.transform_aggregate(corr_value='max(corr_value)')
    .transform_calculate(label="'Pearson r = ' + format(datum.corr_value, '.2f')")
    .mark_text(align='left', baseline='top', fontSize=13)
    .encode(x=alt.value(10), y=alt.value(30), text='label:N')
)

chart7 = (
    (points7 + trend7 + r2_text7 + corr_text7)
    .add_params(country_select7)
    .properties(width=700, height=420,
                title=alt.TitleParams('Residential Mobility vs Stringency Index by Country',
                                      fontSize=13, fontWeight=600))
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridOpacity=0.12, labelFontSize=11, titleFontSize=12)
)

# ── Charts 8 & 9: ML models ───────────────────────────────────────────────────
label_map = {
    'stringency_lag_2':                'Policy Stringency (2-wk lag)',
    'new_cases_smoothed_per_million':  'New Cases / Million',
    'aged_65_older':                   'Pop. Aged 65+',
    'human_development_index':         'Human Dev. Index',
    'population_density':              'Population Density',
    'reproduction_rate':               'Reproduction Rate',
    'diabetes_prevalence':             'Diabetes Prevalence',
    'hospital_beds_per_thousand':      'Hospital Beds / 1k',
    'people_vaccinated_per_hundred':   'Vaccination Rate',
    'life_expectancy':                 'Life Expectancy',
}

# Fit scaled OLS on full data for coefficient chart
scaler_all = StandardScaler()
X_all_sc   = scaler_all.fit_transform(df_model[features].values)
lr_all     = LinearRegression().fit(X_all_sc, df_model['new_deaths_smoothed_per_million'].values)

coef_df = pd.DataFrame({
    'Feature':     [label_map.get(f, f) for f in features],
    'Coefficient': lr_all.coef_,
}).sort_values('Coefficient')
coef_df['Direction'] = coef_df['Coefficient'].apply(
    lambda x: 'Increases Deaths' if x > 0 else 'Decreases Deaths')

chart8 = (
    alt.Chart(coef_df).mark_bar(size=18)
    .encode(
        x=alt.X('Coefficient:Q', title='Standardized Coefficient (OLS)'),
        y=alt.Y('Feature:N',
                sort=alt.EncodingSortField(field='Coefficient', order='ascending'),
                title=''),
        color=alt.Color('Direction:N',
                        scale=alt.Scale(domain=['Increases Deaths', 'Decreases Deaths'],
                                        range=['#e45756', '#4c78a8']),
                        legend=alt.Legend(title='Effect on Mortality', orient='bottom')),
        tooltip=[alt.Tooltip('Feature:N'), alt.Tooltip('Coefficient:Q', format='.3f')]
    )
    .properties(width=700, height=350,
                title=alt.TitleParams('What Predicts COVID Mortality? (Scaled OLS Coefficients)',
                                      fontSize=13, fontWeight=600))
    .configure_view(strokeWidth=0)
    .configure_axis(labelFontSize=11, titleFontSize=12)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

# Time-based split (80 / 20 by date — correct for time-series)
split_date9 = df_model['week'].quantile(0.8)
train9 = df_model[df_model['week'] <= split_date9]
test9  = df_model[df_model['week'] >  split_date9]

X_tr_raw = train9[features].values;  X_te_raw = test9[features].values
y_tr9    = train9['new_deaths_smoothed_per_million'].values
y_te9    = test9['new_deaths_smoothed_per_million'].values

scaler9  = StandardScaler()
X_tr_sc  = scaler9.fit_transform(X_tr_raw)
X_te_sc  = scaler9.transform(X_te_raw)

lr_simple9 = LinearRegression().fit(X_tr_sc[:, [0]], y_tr9)
lr_full9   = LinearRegression().fit(X_tr_sc, y_tr9)
rf9        = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_raw, y_tr9)

r2_df9 = pd.DataFrame([
    {'Model': 'OLS: Stringency Only', 'r2': r2_score(y_te9, lr_simple9.predict(X_te_sc[:, [0]])), 'Type': 'Linear'},
    {'Model': 'OLS: Full Features',   'r2': r2_score(y_te9, lr_full9.predict(X_te_sc)),           'Type': 'Linear'},
    {'Model': 'Random Forest',        'r2': r2_score(y_te9, rf9.predict(X_te_raw)),               'Type': 'Tree'},
])

r2_chart9 = (
    alt.Chart(r2_df9).mark_bar(size=40)
    .encode(
        x=alt.X('Model:N', sort='-y', title='',
                 axis=alt.Axis(labelAngle=-15)),
        y=alt.Y('r2:Q', title='R² Score (held-out test set)'),
        color=alt.Color('Type:N',
                        scale=alt.Scale(domain=['Linear', 'Tree'],
                                        range=['#4c78a8', '#f28e2b']),
                        legend=alt.Legend(title='Model Type')),
        tooltip=[alt.Tooltip('Model:N'), alt.Tooltip('r2:Q', title='R²', format='.3f')]
    )
    .properties(width=750, height=280,
                title=alt.TitleParams('Model R² Comparison (Time-Based Split)',
                                      fontSize=13, fontWeight=600))
)

rf_imp_df = pd.DataFrame({
    'Feature':    [label_map.get(f, f) for f in features],
    'Importance': rf9.feature_importances_,
}).sort_values('Importance')

rf_imp_chart = (
    alt.Chart(rf_imp_df).mark_bar(size=18, color='#f28e2b')
    .encode(
        x=alt.X('Importance:Q', title='Feature Importance'),
        y=alt.Y('Feature:N',
                sort=alt.EncodingSortField(field='Importance', order='ascending'),
                title=''),
        tooltip=[alt.Tooltip('Feature:N'), alt.Tooltip('Importance:Q', format='.3f')]
    )
    .properties(width=750, height=300,
                title=alt.TitleParams('Random Forest Feature Importance',
                                      fontSize=13, fontWeight=600))
)

r2_zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='black', strokeWidth=1).encode(y='y:Q')
r2_labels = (
    alt.Chart(r2_df9).mark_text(dy=-8, fontSize=11, fontWeight=600)
    .encode(
        x=alt.X('Model:N', sort=alt.EncodingSortField(field='r2', order='descending')),
        y='r2:Q',
        text=alt.Text('r2:Q', format='.3f'),
        color=alt.condition('datum.r2 >= 0', alt.value('#1a1a2e'), alt.value('#e45756'))
    )
)
r2_chart9 = (r2_chart9 + r2_zero + r2_labels)

chart9 = (
    alt.vconcat(r2_chart9, rf_imp_chart)
    .configure_view(strokeWidth=0)
    .configure_axis(labelFontSize=11, titleFontSize=12)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

# ══════════════════════════════════════════════════════════════════════════════
# ASSEMBLE TAB WIDGET  (9 tabs)
# ══════════════════════════════════════════════════════════════════════════════

tabs_info = [
    ('🗺️  Deaths Map',
     'Cumulative COVID-19 deaths per million population (log scale). Sets the stage — where did the pandemic hit hardest?',
     chart1),
    ('📈  Stringency Over Time',
     'Monthly Oxford Stringency Index 2020–2022. Click a country in the legend to highlight it.',
     chart2),
    ('🎬  Animated Stringency Map',
     'Watch policy stringency evolve globally month by month. Press play or drag the slider.',
     fig4),
    ('📉  Policy & Mortality Trends',
     'Weekly stringency and mortality for 8 peer nations side by side. Click legend to isolate a country.',
     chart5),
    ('⏱️  Policy Timing vs Mortality',
     'Did acting faster before the first death reduce mortality? Drag to select a group — the histogram updates to show their death % distribution.',
     chart3),
    ('🚶  Residential Mobility',
     'Did policies actually change behavior? Smoothed residential mobility over time — grey = all countries, colored = key nations.',
     chart6),
    ('🔗  Stringency vs Mobility',
     'The direct relationship: stricter policies → more time at home? Use the dropdown to explore each country.',
     chart7),
    ('📊  OLS Coefficients',
     'Controlling for all factors simultaneously — which variables actually drive COVID mortality up or down?',
     chart8),
    ('🤖  Model Comparison',
     'Time-based 80/20 split — OLS vs Random Forest on held-out future data. Note: negative R² on OLS means the linear model fails to generalize to future time periods, while Random Forest adapts better.',
     chart9),
]
out_widgets = []
for title, desc, chart in tabs_info:
    out = widgets.Output()
    with out:
        display(HTML(f'<div class="section-header">{title.strip()}</div>'
                     f'<div class="section-desc">{desc}</div>'))
        if hasattr(chart, 'show'):
            chart.show()
        else:
            display(chart)
    out_widgets.append(out)

tab = widgets.Tab(children=out_widgets)
for i, (title, _, _) in enumerate(tabs_info):
    tab.set_title(i, title)

display(tab)